# Ekstrak Audio to Text

In [1]:
# Whisper resmi (stabil):
!pip install -U openai-whisper --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 67.0 MB/s

In [2]:
import os, re, sys, json, time, subprocess
from pathlib import Path
from typing import List, Dict

import pandas as pd
from tqdm.auto import tqdm

# ================== KONFIGURASI ==================
# Lokasi pencarian audio (otomatis semua subfolder /kaggle/input)
AUDIO_ROOTS: List[Path] = [Path("/kaggle/input/d/eldintarofarrandi/satria-data/audio-new/audio-new/train")]

# Ekstensi audio yang diproses
AUDIO_EXTS = {".mp3", ".wav", ".m4a", ".flac", ".ogg", ".aac"}

# File label opsional (akan dicari di /kaggle/input)
CANDIDATE_LABEL_FILES = [
    "label-datatrain/cleaned_datatrain (1) (1).csv",
]

# Whisper
WHISPER_MODEL = "large-v3-turbo"      # tiny|base|small|medium|large-v3
WHISPER_LANGUAGE = None      # None = auto, atau "indonesian"
USE_FASTER_WHISPER = False   # True untuk faster-whisper (jika diinstal)

# Output
SAVE_DIR = Path("/kaggle/working")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
PROGRESS_CSV = SAVE_DIR / "progress_transcripts.csv"
OUTPUT_CSV   = SAVE_DIR / "transcripts.csv"

# =================================================

def ensure_ffmpeg():
    """Pastikan ffmpeg tersedia; fallback imageio-ffmpeg jika belum ada."""
    try:
        subprocess.check_output(["ffmpeg", "-version"], stderr=subprocess.STDOUT)
        return
    except Exception:
        try:
            import imageio_ffmpeg as iio_ffmpeg
            exe = iio_ffmpeg.get_ffmpeg_exe()
            os.environ["PATH"] = str(Path(exe).parent) + os.pathsep + os.environ.get("PATH", "")
            subprocess.check_output(["ffmpeg", "-version"], stderr=subprocess.STDOUT)
        except Exception as e:
            raise EnvironmentError(
                "FFmpeg tidak ditemukan dan gagal fallback imageio-ffmpeg. "
                "Pastikan cell install sudah dijalankan."
            ) from e

def discover_audio(roots: List[Path]) -> List[Path]:
    files = []
    for root in roots:
        if not root.exists():
            continue
        for r, _, fs in os.walk(root):
            for f in fs:
                p = Path(r) / f
                if p.suffix.lower() in AUDIO_EXTS:
                    files.append(p)
    return sorted(files)

def load_optional_labels() -> Dict[str, str]:
    """Cari file label opsional di /kaggle/input dan kembalikan dict id->emotion."""
    label_df = None
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f in CANDIDATE_LABEL_FILES:
                path = Path(root) / f
                try:
                    tmp = pd.read_csv(path)
                except Exception as e:
                    print("Gagal baca label:", path, "->", e)
                else:
                    label_df = tmp
                    print("Label file ditemukan:", path)
                break
        if label_df is not None:
            break

    if label_df is None or "id" not in label_df.columns or "emotion" not in label_df.columns:
        return {}

    label_df = label_df.copy()
    label_df["id"] = label_df["id"].astype(str)
    return dict(zip(label_df["id"], label_df["emotion"]))

def id_from_filename(p: Path) -> str:
    """Ekstrak id dari nama file (ex: 101.mp3 -> 101). Jika tidak ada angka, pakai nama file."""
    s = p.stem
    m = re.search(r"\d+", s)
    return m.group(0) if m else s

# --------- Inisialisasi lingkungan ---------
ensure_ffmpeg()
audio_files = discover_audio(AUDIO_ROOTS)
print(f"Audio ditemukan: {len(audio_files)} file")
if not audio_files:
    raise FileNotFoundError("Tidak ditemukan file audio di /kaggle/input. "
                            "Pastikan dataset audio sudah di-add ke notebook.")

label_map = load_optional_labels()
if label_map:
    print(f"Label tersedia untuk {len(label_map)} id")

# --------- Progress helpers ---------
if PROGRESS_CSV.exists():
    prog = pd.read_csv(PROGRESS_CSV, dtype="object")
else:
    prog = pd.DataFrame(
        columns=["id", "audio_path", "text_transcript", "status", "error", "emotion"]
    ).astype("object")

def already_done(path: Path) -> bool:
    if prog.empty:
        return False
    r = prog[prog["audio_path"] == str(path)]
    return (len(r) and str(r.iloc[0]["status"]) == "done")

def save_progress(rec: dict):
    global prog
    # pastikan semua kolom ada
    for k in ["id", "audio_path", "text_transcript", "status", "error", "emotion"]:
        rec.setdefault(k, None)
    mask = (prog["audio_path"] == rec["audio_path"])
    if mask.any():
        for k, v in rec.items():
            prog.loc[mask, k] = "" if (v is None and k != "emotion") else v
    else:
        row = {k: ("" if (v is None and k != "emotion") else v) for k, v in rec.items()}
        prog = pd.concat([prog, pd.DataFrame([row])[prog.columns]], ignore_index=True)
    prog.to_csv(PROGRESS_CSV, index=False)

# --------- Init model Whisper ---------
import torch
if USE_FASTER_WHISPER:
    try:
        from faster_whisper import WhisperModel
    except Exception as e:
        raise ImportError("faster-whisper belum terpasang. Jalankan: `!pip -q install -U faster-whisper`") from e
    device = "cuda" if torch.cuda.is_available() else "cpu"
    compute_type = "float16" if torch.cuda.is_available() else "int8"
    fw_model = WhisperModel(WHISPER_MODEL, device=device, compute_type=compute_type)
    print(f"Loaded faster-whisper {WHISPER_MODEL} | device={device} | compute_type={compute_type}")
else:
    import whisper
    WHISPER_MODEL = "base"  # bisa "tiny", "small", "medium", "large"
    ow_model = whisper.load_model(WHISPER_MODEL)
    print(f"Loaded openai-whisper {WHISPER_MODEL} | CUDA={torch.cuda.is_available()}")


# --------- Transkripsi loop (resume-safe) ---------
for fp in tqdm(audio_files):
    if already_done(fp):
        continue

    _id = id_from_filename(fp)
    emo = label_map.get(str(_id)) if label_map else None
    rec = {"id": str(_id), "audio_path": str(fp), "text_transcript": "", "status": "started", "error": "", "emotion": emo}
    save_progress(rec)

    try:
        if USE_FASTER_WHISPER:
            segs, info = fw_model.transcribe(str(fp), language=WHISPER_LANGUAGE, vad_filter=True)
            text = " ".join(seg.text.strip() for seg in segs)
        else:
            args = {"fp16": torch.cuda.is_available()}
            if WHISPER_LANGUAGE is not None:
                args["language"] = WHISPER_LANGUAGE
            result = ow_model.transcribe(str(fp), **args)
            text = result.get("text", "").strip()

        rec["text_transcript"] = text
        rec["status"] = "done"
        rec["error"] = ""
        save_progress(rec)

    except Exception as e:
        rec["status"] = "failed"
        rec["error"] = str(e)
        save_progress(rec)

# --------- Tulis hasil akhir ---------
final = pd.read_csv(PROGRESS_CSV, dtype="object")
out = final[final["status"] == "done"][["id", "audio_path", "text_transcript", "emotion"]].copy()
out.to_csv(OUTPUT_CSV, index=False)
print("Selesai →", OUTPUT_CSV, "| rows:", len(out))

display(out.head())


Audio ditemukan: 779 file


100%|███████████████████████████████████████| 139M/139M [00:09<00:00, 16.1MiB/s]


Loaded openai-whisper base | CUDA=True


  0%|          | 0/779 [00:00<?, ?it/s]

Selesai → /kaggle/working/transcripts.csv | rows: 779


,id,audio_path,text_transcript,emotion
0,1,/kaggle/input/d/eldintarofarrandi/satria-data/...,Tau ga rumah tradisional laos kayak gimana? Ka...,NaN
1,10,/kaggle/input/d/eldintarofarrandi/satria-data/...,"Jadi selama kita di Thailand, Overland, kita t...",NaN
2,100,/kaggle/input/d/eldintarofarrandi/satria-data/...,spatulanya kan kita mendasihkannya adopted unt...,NaN
3,101,/kaggle/input/d/eldintarofarrandi/satria-data/...,Ipin apa kamu boleh located Aduhumph система b...,NaN
4,102,/kaggle/input/d/eldintarofarrandi/satria-data/...,"Saya rasa apa yang kita bisa melakukan adalah,...",NaN


In [3]:
# import os
# import pandas as pd

# df_t = pd.read_csv("/kaggle/input/transcript-satria-data/transcripts.csv")
# df_clean = pd.read_csv("/kaggle/input/datatrain-bersih/cleaned_datatrain (1) (1).csv")

In [4]:
# df_t.drop_duplicates()

In [5]:
# id_col = df_clean['id']

In [6]:
# df_clean

In [7]:
# df = pd.merge(df_clean[['id', 'emotion']], df_t[['id', 'text_transcript']], on = 'id', how='inner')

In [8]:
# df.tail()

In [9]:
# order_ids = df_clean['id'].tolist()

In [10]:
# df['id'] = pd.Categorical(df['id'], categories=order_ids, ordered=True)
# df = df.sort_values('id').reset_index(drop=True)

In [11]:
# df.info()

In [12]:
# df.isna().sum()

In [13]:
# df = df.dropna()

In [14]:
# df.to_csv("hasil_clean.csv")

In [15]:
# df = pd.read_csv("/kaggle/working/hasil_clean.csv")

In [16]:
# df

In [17]:
# import re
# import unicodedata

# class IndonesianTextPreprocessor:
#     """
#     Preprocessor untuk transcript Instagram Reels.
#     Lebih hati-hati agar teks normal tidak terhapus.
#     """

#     def __init__(self, keep_hashtag_text=True):
#         self.keep_hashtag_text = keep_hashtag_text

#         # slang → baku
#         self.slang_dict = {
#             'ga':'tidak','gak':'tidak','ngga':'tidak','nggak':'tidak','ngk':'tidak',
#             'banget':'sangat','bgt':'sangat','nih':'ini','dong':'','tau':'tahu',
#             'udah':'sudah','aja':'saja','gitu':'begitu','dr':'dari','btw':'ngomong-ngomong',
#             'pls':'tolong','pls.':'tolong','gw':'saya','gue':'saya','loe':'kamu','lu':'kamu',
#             'tp':'tapi','yg':'yang','jd':'jadi','krn':'karena','dgn':'dengan'
#         }

#         self.preserve_words = {'BYD','WAN','GIAS','Mitsubishi','Nissan','OEM','solar guard'}
#         self.filler_words   = {'uf','um','uh','eh','ah','emm','hmm','hehe','hehehe','haha','hahaha','aduuh','waduh'}

#         self.music_markers = {'music','musik','backsound','bgm','instrumental','remix','beat',
#                               'intro music','outro music','lagu','song','lyrics','lirik','chorus','verse'}
#         self.sfx_markers   = {'laugh','laughter','applause','cheer','sigh','clap','drumroll'}
#         self.lyrics_fillers= {'la','lala','lalala','na','nana','nanana','yeah','yo','woo','wooo','woah','hey','ooh','aa','aaa'}

#         # regex util
#         self.re_url      = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
#         self.re_mention  = re.compile(r'@\w+')
#         self.re_hashtag  = re.compile(r'#(\w{2,})')
#         self.re_timestamp= re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b')
#         self.re_brackets = re.compile(r'[\[\(]{1}[^)\]]{0,40}\b(music|musik|song|lyrics|lirik|laugh|applause|cheer|sfx|beat)\b[^)\]]{0,40}[\)\]]', re.IGNORECASE)
#         self.re_notes    = re.compile(r'\b(copyright|all rights|credits|dm for|link in bio|subscribe|follow)\b', re.IGNORECASE)
#         self.re_repeat   = re.compile(r'(.)\1{2,}', re.UNICODE)
#         self.re_multi_ws = re.compile(r'\s+')

#     @staticmethod
#     def strip_emojis(text: str) -> str:
#         return ''.join(ch for ch in text if not unicodedata.category(ch).startswith('So'))

#     def normalize_hashtags(self, text: str) -> str:
#         return self.re_hashtag.sub(lambda m: m.group(1) if self.keep_hashtag_text else ' ', text)

#     def clean_platform_artifacts(self, text: str) -> str:
#         t = self.re_url.sub(' ', text)
#         t = self.re_mention.sub(' ', t)
#         t = self.normalize_hashtags(t)
#         t = self.re_timestamp.sub(' ', t)
#         t = self.re_brackets.sub(' ', t)
#         t = self.re_notes.sub(' ', t)
#         t = self.strip_emojis(t)
#         return t

#     def normalize_slang_and_repetitions(self, text: str) -> str:
#         text = self.re_repeat.sub(r'\1\1', text)
#         toks, out = text.split(), []
#         for w in toks:
#             if w in self.preserve_words:
#                 out.append(w)
#                 continue
#             lw = w.lower()
#             if lw in self.slang_dict:
#                 out.append(self.slang_dict[lw])
#             elif lw in self.filler_words:
#                 continue
#             else:
#                 out.append(w)
#         return ' '.join(out)

#     def final_cleanup(self, text: str) -> str:
#         t = re.sub(r'[^\w\s]', ' ', text)
#         t = self.re_multi_ws.sub(' ', t).strip().lower()
#         return t

#     # deteksi musik/noise (lebih longgar, tidak pakai rule "too_short")
#     def is_music_or_noise(self, text: str) -> bool:
#         low = text.lower()
#         music_hits = sum(int(k in low) for k in self.music_markers)
#         sfx_hits   = sum(int(k in low) for k in self.sfx_markers)
#         toks = [t for t in re.findall(r'\w+', low)]
#         if not toks:
#             return True
#         lyric_ratio = sum(t in self.lyrics_fillers for t in toks) / max(1, len(toks))
#         return (music_hits + sfx_hits >= 1) or (lyric_ratio >= 0.6)

#     def preprocess(self, text: str, return_flag=False):
#         if not isinstance(text, str):
#             return "" if not return_flag else ("", True)

#         t = self.clean_platform_artifacts(text)
#         t = self.normalize_slang_and_repetitions(t)
#         t = self.final_cleanup(t)

#         flag_music = self.is_music_or_noise(t)
#         if return_flag:
#             return t, flag_music
#         return t

#     def preprocess_batch(self, texts, return_flag=False):
#         return [self.preprocess(x, return_flag=return_flag) for x in texts]


In [18]:
# pre = IndonesianTextPreprocessor()

# # simpan teks + flag
# df['text_cleaned'], df['is_music_only'] = zip(*df['text_transcript'].map(lambda x: pre.preprocess(x, return_flag=True)))

# # semua id tetap ada
# print(df.loc[df['id']==1])

In [19]:
# df

In [20]:
# df.to_csv("hasil_clean2.csv")

In [21]:
# df

In [22]:
# import matplotlib.pyplot as plt

# # hitung jumlah tiap label
# label_counts = df['emotion'].value_counts().sort_index()

# # plot
# plt.figure(figsize=(8,5))
# plt.bar(label_counts.index, label_counts.values, color='skyblue')
# plt.xticks(rotation=45, ha='right')
# plt.xlabel("Emotion Label")
# plt.ylabel("Jumlah")
# plt.title("Distribusi Label Emotion")
# plt.tight_layout()
# plt.show()

# # kalau mau lihat juga angkanya:
# print(label_counts)


In [23]:
# df.info()

# Normalisasi

In [24]:
# # Jalankan sekali
# !pip -q install symspellpy

In [25]:
# import re
# import pandas as pd

# # ---- dari jawaban sebelumnya ----
# import nltk
# try: nltk.data.find("corpora/wordnet")
# except LookupError: nltk.download("wordnet")
# try: nltk.data.find("corpora/omw-1.4")
# except LookupError: nltk.download("omw-1.4")
# try: nltk.data.find("corpora/stopwords")
# except LookupError: nltk.download("stopwords")

# from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer

# EN_STOP = set(stopwords.words("english"))
# EN_LEM  = WordNetLemmatizer()

# # extend misspell map
# MISSPELL_MAP=({
#     "lagi": "sedang",
#     "boong": "bohong",
#     "rembut": "rambut",
#     "kepo": "ingin tahu",
#     "gimana": "bagaimana",
#     "kelar": "selesai",
#     "ngangkat": "mengangkat",
#     "mentarik": "menarik",
#     "lagapasar": "lapangan",
#     "kemanis": "kemana",
#     "de-eskpresikan": "diekspresikan",
#     "menantilkan": "menantikan",
#     "ngobil": "mobil",
#     "mode": "model",
#     "go": "gokil",    # hati2: "go" sering normal word, kalau ini khusus slang -> ganti mapping
#     "kil": "gokil",   # biar "go kil" -> "gokil"
#     "usian": "usia",
#     "benzin": "bensin",
#     "ganting": "genting",
#     "mobin": "mobil",
#     "irrit": "irit",
#     "kelarifikasi": "klarifikasi",
#     "dipaman": "dipamerkan",    # jika ada "dipaman kan" -> akan jadi dua token, bisa normalisasi via regex juga
#     "kambosya": "kamboja",
#     "kapra": "kaprah",
#     "mark": "mari",
#     "jen": "jangan",
#     "penutantangan": "penuh tantangan",
#     "sulaap": "sulap",
#     "teksjur": "tekstur",
#     "kesantikan": "kecantikan",
#     "perseratan": "persyaratan",
# })

# MULTI_MAP = {
#     "permit":"izin","marketing":"pemasaran","product":"produk","car":"mobil","cars":"mobil",
#     "happy":"senang","fun":"seru","fear":"takut","anger":"marah","sad":"sedih","joy":"senang",
#     "trust":"percaya","proud":"bangga"
# }

# TOKEN_RE = re.compile(r"\w+")

# def normalize_token(tok: str):
#     t = tok.lower()

#     # a) rule-based typo fix
#     if t in MISSPELL_MAP:
#         return MISSPELL_MAP[t], "typo_fixed"

#     # b) English handling
#     is_ascii = all(ord(c) < 128 for c in t)
#     if is_ascii:
#         lemma = EN_LEM.lemmatize(t)
#         if lemma in MULTI_MAP:
#             return MULTI_MAP[lemma], "translated_en"
#         if t in EN_STOP:
#             return "", "drop_en_stop"
#         if lemma != t:
#             return lemma, "en_lemmatized"

#     return t, ""

# def normalize_text_series(series: pd.Series):
#     normalized_text, had_changes, had_multilingual, unresolved_typos = [], [], [], []
#     uncertain_list = {"diungkannya", "duan", "nacar"}

#     for text in series.astype(str):
#         tokens = TOKEN_RE.findall(text)
#         new_toks, changed, multi, unresolved = [], False, False, []

#         for tok in tokens:
#             new_tok, how = normalize_token(tok)
#             if how: changed = True
#             if how == "translated_en": multi = True
#             if tok.lower() in uncertain_list: unresolved.append(tok)
#             if new_tok != "": new_toks.append(new_tok)

#         normalized_text.append(" ".join(new_toks).strip())
#         had_changes.append(changed)
#         had_multilingual.append(multi)
#         unresolved_typos.append(", ".join(unresolved) if unresolved else "")

#     return (pd.Series(normalized_text, index=series.index),
#             pd.Series(had_changes, index=series.index, dtype=bool),
#             pd.Series(had_multilingual, index=series.index, dtype=bool),
#             pd.Series(unresolved_typos, index=series.index))

# # ==== langkah normalisasi awal (tanpa spellcheck) ====
# df['text_normalized'], df['had_changes'], df['had_multilingual'], df['unresolved_tokens'] = normalize_text_series(df['text_cleaned'])

# print("Contoh sebelum spellcheck:")
# print(df[['text_cleaned','text_normalized','unresolved_tokens']].head(3))


In [26]:
# # ===== SymSpell setup =====
# from symspellpy import SymSpell, Verbosity
# from collections import Counter
# import math

# # 1) Bangun kamus dari korpus (pakai text_normalized agar bersih)
# def build_symspell_from_corpus(texts: pd.Series,
#                                max_dictionary_edit_distance=2,
#                                prefix_length=7,
#                                min_freq=3,
#                                extra_whitelist=None):
#     """
#     - texts: pd.Series string
#     - min_freq: token dengan frekuensi < min_freq tidak dimasukkan
#     """
#     sym = SymSpell(max_dictionary_edit_distance=max_dictionary_edit_distance,
#                    prefix_length=prefix_length)

#     # hitung frekuensi token
#     freq = Counter()
#     for t in texts.astype(str):
#         for tok in TOKEN_RE.findall(t.lower()):
#             freq[tok] += 1

#     # tambahkan ke dictionary
#     for tok, c in freq.items():
#         if c >= min_freq:
#             sym.create_dictionary_entry(tok, c)

#     # whitelist agar TIDAK dikoreksi (masukkan sebagai entry frekuensi besar)
#     whitelist = set(extra_whitelist or [])
#     for w in whitelist:
#         sym.create_dictionary_entry(w.lower(), max(freq.values()) + 1000)

#     return sym

# # 2) daftar kata yang tidak boleh dikoreksi (brand/istilah)
# WHITELIST = {
#     "byd","wan","gias","mitsubishi","nissan","oem","solar","guard","pellek","pepaya"
# }

# # build symspell dari korpus
# symspell = build_symspell_from_corpus(df['text_normalized'], min_freq=2, extra_whitelist=WHITELIST)

# def symspell_correct_token(tok: str, symspell: SymSpell,
#                            max_edit_distance=2, suggest_verbosity=Verbosity.TOP):
#     """
#     Kembalikan token terkoreksi (atau token asli jika tak ada yang lebih baik).
#     """
#     lower = tok.lower()
#     # skip angka atau 1 huruf
#     if lower.isdigit() or len(lower) == 1:
#         return tok, False

#     # skip whitelist
#     if lower in WHITELIST or lower in MISSPELL_MAP or lower in MULTI_MAP:
#         return tok, False

#     suggestions = symspell.lookup(lower, suggest_verbosity, max_edit_distance=max_edit_distance)
#     if suggestions:
#         best = suggestions[0].term
#         if best != lower:
#             return best, True
#     return tok, False

# def symspell_correct_series(series: pd.Series, symspell: SymSpell):
#     corrected_text, had_spellchange = [], []
#     for text in series.astype(str):
#         toks = TOKEN_RE.findall(text)
#         new_toks, changed = [], False
#         for t in toks:
#             c, ch = symspell_correct_token(t, symspell)
#             new_toks.append(c)
#             changed = changed or ch
#         corrected_text.append(" ".join(new_toks))
#         had_spellchange.append(changed)
#     return pd.Series(corrected_text, index=series.index), pd.Series(had_spellchange, index=series.index, dtype=bool)

# # 3) apply spell-check
# df['text_spell'], df['had_spellchange'] = symspell_correct_series(df['text_normalized'], symspell)

# print("\nContoh setelah spellcheck:")
# print(df[['text_normalized','text_spell','had_spellchange']].head(5))


In [27]:
# df['text_final'] = df['text_spell'].where(df['text_spell'].str.len() > 0, df['text_normalized'])

# # Laporan ringkas
# print("\nRingkasan perubahan:")
# print("Baris yang berubah karena normalisasi:", int(df['had_changes'].sum()))
# print("Baris yang ada multilingual (diterjemahkan):", int(df['had_multilingual'].sum()))
# print("Baris yang berubah karena spellcheck:", int(df['had_spellchange'].sum()))

# # OPTIONAL: review token yang tak pasti (dari list kamu)
# print("\nContoh unresolved tokens (untuk ditinjau & tambahkan ke MISSPELL_MAP jika perlu):")
# print(df.loc[df['unresolved_tokens'].str.len()>0, ['text_cleaned','text_normalized','unresolved_tokens']].head(10))

In [28]:
# df

In [29]:
# df['unresolved_tokens'].unique()

# Stopwords

In [30]:
# !pip install Sastrawi

In [31]:
# import re
# import nltk
# from nltk.corpus import stopwords
# from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# # pastikan stopwords NLTK sudah ada
# try:
#     nltk.data.find("corpora/stopwords")
# except LookupError:
#     nltk.download("stopwords")

# # --- STOPWORDS LIST ---
# # Bahasa Inggris dari NLTK
# nltk_stop_en = set(stopwords.words("english"))

# # Bahasa Indonesia dari Sastrawi
# factory = StopWordRemoverFactory()
# sastrawi_stop_id = set(factory.get_stop_words())

# # Domain / custom stopwords
# custom_stopwords = {"mobil", "produk"}  # contoh

# # Gabungan
# ALL_STOPWORDS = nltk_stop_en.union(sastrawi_stop_id).union(custom_stopwords)

# # --- TOKENIZER ---
# TOKEN_RE = re.compile(r"\w+")

# def remove_stopwords(text: str, stopwords_set=None) -> str:
#     if not isinstance(text, str):
#         return ""
#     if stopwords_set is None:
#         stopwords_set = ALL_STOPWORDS
#     tokens = TOKEN_RE.findall(text.lower())
#     filtered = [t for t in tokens if t not in stopwords_set]
#     return " ".join(filtered)

# # --- APPLY ke DataFrame ---
# df["text_nostop"] = df["text_final"].apply(lambda x: remove_stopwords(x, ALL_STOPWORDS))

# # cek hasil
# print(df[["text_final", "text_nostop"]].head(10))

# Tokenizer

In [32]:
# # !pip install Sastrawi wordcloud

# import re
# import math
# import numpy as np
# import matplotlib.pyplot as plt
# from wordcloud import WordCloud, STOPWORDS

# # ====== 0) Label column ======
# LABEL_COL = 'emotion_standardized' if 'emotion_standardized' in df.columns else 'emotion'
# assert LABEL_COL in df.columns, f"Kolom label '{LABEL_COL}' tidak ditemukan."

# # ====== 1) Frasa domain yang ingin digabung (bigram/trigram) ======
# # contoh: "solar guard" → solar_guard; tambahkan sesuai kebutuhanmu
# DOMAIN_PHRASES = [
#     r"solar guard",
#     r"pipi f",
#     r"harga mobil",
#     r"test drive",
# ]

# def fuse_phrases(text: str, phrases):
#     if not isinstance(text, str):
#         return ""
#     t = text
#     for p in phrases:
#         # ganti spasi dengan underscore untuk frasa lengkap (case-insensitive)
#         pattern = re.compile(rf"\b{p}\b", flags=re.IGNORECASE)
#         t = pattern.sub(lambda m: m.group(0).lower().replace(" ", "_"), t)
#     return t

# df['text_fused'] = df['text_nostop'].astype(str).apply(lambda s: fuse_phrases(s, DOMAIN_PHRASES))

# # ====== 2) Stemming Sastrawi untuk Bahasa Indonesia ======
# from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
# stemmer = StemmerFactory().create_stemmer()

# TOKEN_RE = re.compile(r"\w+")
# def stem_token_id(tok: str) -> str:
#     # jangan ubah token frasa yang sudah di-underscore
#     if "_" in tok:
#         return tok.lower()
#     return stemmer.stem(tok.lower())

# df['tokenized_text'] = df['text_fused'].apply(
#     lambda s: [stem_token_id(t) for t in TOKEN_RE.findall(s)]
# )

# # ====== 3) Stopwords gabungan: Sastrawi (ID) + NLTK (EN) + custom domain ======
# import nltk
# from nltk.corpus import stopwords
# # pastikan corpus ada
# try: nltk.data.find("corpora/stopwords")
# except LookupError: nltk.download("stopwords")

# # Sastrawi stopwords
# from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
# sastrawi_stop_id = set(StopWordRemoverFactory().get_stop_words())

# # NLTK English stopwords
# nltk_stop_en = set(stopwords.words("english"))

# # Custom domain stopwords (tambahkan sesuai kebutuhanmu)
# custom_stop = {
#     "aja","nih","dong","banget","bgt","produk","mobil","brand","promo",
#     "ga","gak","ngga","nggak","yah","ya","sih","lho","loh","deh","lah",
#     # jika ingin menghapus kata fungsi umum ID (beberapa sudah ada di Sastrawi)
#     "yang","dan","di","ke","dari","ini","itu","untuk","pada","dengan",
# }

# ALL_STOPWORDS = sastrawi_stop_id.union(nltk_stop_en).union(custom_stop)

# # filter stopwords dari token
# df['tokenized_text'] = df['tokenized_text'].apply(
#     lambda toks: [t for t in toks if t not in ALL_STOPWORDS and len(t) > 1]
# )

# # ====== 4) Siapkan korpus per label untuk WordCloud ======
# def tokens_to_text(tokens):
#     # gabung kembali ke string (wordcloud butuh string)
#     return " ".join(tokens)

# texts_per_label = (
#     df[[LABEL_COL, 'tokenized_text']]
#     .groupby(LABEL_COL)['tokenized_text']
#     .apply(lambda lists: tokens_to_text([t for lst in lists for t in lst]))
#     .to_dict()
# )

# # ====== 5) WordCloud per label ======
# # gabungkan stopwords bawaan WordCloud agar lebih bersih
# wc_stop = STOPWORDS.union(set())  # tambahkan jika ada kata tambahan: .union({"kata"})

# labels = sorted(texts_per_label.keys())
# n = len(labels)
# ncols = 4 if n >= 4 else n
# nrows = math.ceil(n / ncols)

# fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.2*ncols, 3.8*nrows))
# if nrows == 1 and ncols == 1:
#     axes = np.array([[axes]])
# elif nrows == 1:
#     axes = np.array([axes])
# elif ncols == 1:
#     axes = axes.reshape(-1, 1)

# for idx, label in enumerate(labels):
#     r, c = divmod(idx, ncols)
#     ax = axes[r, c]
#     blob = texts_per_label[label]

#     if not isinstance(blob, str) or not blob.strip():
#         ax.axis('off')
#         ax.set_title(f"{label} (no text)", fontsize=11)
#         continue

#     wc = WordCloud(
#         width=900, height=600,
#         background_color="white",
#         stopwords=wc_stop,
#         collocations=False,   # kita sudah gabungkan frasa manual (underscore)
#         max_words=200
#     ).generate(blob)

#     ax.imshow(wc, interpolation='bilinear')
#     ax.axis('off')
#     ax.set_title(label, fontsize=12)

# # matikan axes kosong
# for j in range(n, nrows*ncols):
#     r, c = divmod(j, ncols)
#     axes[r, c].axis('off')

# plt.tight_layout()
# plt.show()

# # (opsional) simpan file per label
# # for label, blob in texts_per_label.items():
# #     if isinstance(blob, str) and blob.strip():
# #         WordCloud(width=1200, height=800, background_color="white", stopwords=wc_stop, collocations=False)\
# #             .generate(blob).to_file(f"wordcloud_{label}.png")


In [33]:
# # ================== CLEANER UNTUK ANALISIS (WORDCLOUD/TF-IDF) ==================
# !pip -q install Sastrawi nltk

# import re, unicodedata, numpy as np, pandas as pd
# from nltk.corpus import stopwords as nltk_stop
# import nltk; nltk.download('stopwords', quiet=True)
# from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
# from sklearn.feature_extraction.text import TfidfVectorizer

# TEXT_COL  = 'text_final'       # atau kolom teks bersihmu
# LABEL_COL =  'emotion'
# TOP_K     = 25

# # --- 1) stopwords dasar + ekstra colloquial/partikel ---
# base_stop = set()
# try:
#     base_stop |= set(nltk_stop.words('indonesian'))
# except: pass

# # partikel, kata ganti, penghubung informal yang sering “noise”
# extra_stop = {
#     # pronomina / deiksis informal
#     'aku','kamu','kalian','gue','gua','lu','loe','dia','kita','mereka','saya',
#     # partikel/pengisi
#     'ya','yah','lah','dong','deh','sih','kan','mah','loh','lho','nih','tuh','nah',
#     # penghubung/umum
#     'jadi','kalau','kalo','kayak','gitu','gtu','gini','begini','begitu','cuma','aja','doang',
#     'sangat','banget','bgt','lebih','biar','buat','bikin','pakai','pake','dapat','bisa',
#     'yang','dan','atau','dengan','tanpa','untuk','dari','ke','di','itu','ini','ada','akan','karena',
#     # yang sering muncul di plotmu:
#     'sama','mau','apa','terus','banyak','satu','punya','langsung','tahu','sekarang','orang',
#     # kampanye/platform
#     'subscribe','follow','link','bio','official',
#     # multilingual kecil
#     'permit'
# }
# stop_all = base_stop | extra_stop

# # --- 2) spelling & slang map (boleh tambah sesuai data) ---
# typo_map = {
#     'paya':'pepaya', 'dipodifikasi':'dimodifikasi', 'vellek':'pellek',
#     'panah':'tanah', 'gubar':'bubar', 'btw':'ngomongngomong',
# }
# slang_map = {
#     'ga':'tidak','gak':'tidak','ngga':'tidak','nggak':'tidak','ngk':'tidak',
#     'udah':'sudah','aja':'saja','bgt':'sangat','dr':'dari','tp':'tapi','yg':'yang','jd':'jadi',
# }

# factory = StemmerFactory(); stemmer = factory.create_stemmer()

# def strip_emoji(s: str) -> str:
#     return ''.join(ch for ch in s if not unicodedata.category(ch).startswith('So'))

# def normalize_basic(s: str) -> str:
#     s = s.lower()
#     s = re.sub(r'https?://\S+|www\.\S+',' ',s)
#     s = re.sub(r'@\w+',' ',s)
#     s = re.sub(r'#',' ',s)                        # hilangkan #, biar teksnya ikut
#     s = re.sub(r'\d+',' ',s)
#     s = strip_emoji(s)
#     s = re.sub(r'[^\w\s]',' ',s)
#     s = re.sub(r'\s+',' ',s).strip()
#     return s

# def replace_map_tokens(tokens, m):
#     return [m.get(t, t) for t in tokens]

# def clean_for_analysis(text: str) -> str:
#     if not isinstance(text, str): return ""
#     t = normalize_basic(text)
#     toks = t.split()
#     toks = replace_map_tokens(toks, typo_map)
#     toks = replace_map_tokens(toks, slang_map)
#     # buang stopwords/partikel + token pendek
#     toks = [w for w in toks if len(w) > 2 and w not in stop_all]
#     # stemming supaya bentuk kata seragam
#     toks = [stemmer.stem(w) for w in toks]
#     # filter lagi setelah stemming
#     toks = [w for w in toks if len(w) > 2 and w not in stop_all]
#     return ' '.join(toks)

# df['_text_for_wc'] = df[TEXT_COL].fillna('').astype(str).apply(clean_for_analysis)

# # --- 3) TF-IDF per label + drop "global terms" (muncul di banyak label) ---
# texts  = df['_text_for_wc'].tolist()
# labels = df[LABEL_COL].astype(str).tolist()

# tfidf = TfidfVectorizer(
#     analyzer='word', ngram_range=(1,2),
#     min_df=2, max_df=0.95, sublinear_tf=True
# )
# X = tfidf.fit_transform(texts)
# vocab = np.array(tfidf.get_feature_names_out())
# labels_ser = pd.Series(labels, name='label')

# # hitung coverage label setiap term (berapa label unik yang mengandung term itu)
# label_mats = []
# for lab, idx in labels_ser.groupby(labels_ser).groups.items():
#     label_mats.append((lab, (X[idx] > 0).sum(axis=0).A1 > 0))
# label_hit = np.zeros(X.shape[1], dtype=int)
# for _, mask in label_mats:
#     label_hit += mask.astype(int)
# label_coverage = label_hit / len(set(labels))
# global_mask = label_coverage >= 0.6   # buang term yang muncul di >=60% label
# keep_cols = np.where(~global_mask)[0]
# Xk = X[:, keep_cols]; vocab_k = vocab[keep_cols]

# # rangkum TOP_K per label (lebih “unik”)
# rows = []
# for lab, idx in labels_ser.groupby(labels_ser).groups.items():
#     mean_vec = Xk[idx].mean(axis=0).A1
#     top_ix = np.argsort(mean_vec)[-TOP_K:][::-1]
#     for r, i in enumerate(top_ix, 1):
#         rows.append({'label': lab, 'rank': r, 'term': vocab_k[i], 'mean_tfidf': float(mean_vec[i])})
# top_df = pd.DataFrame(rows)

# # --- 4) plot ulang ---
# import matplotlib.pyplot as plt
# def plot_top_terms(label, k=15):
#     sub = top_df[top_df['label']==label].nlargest(k,'mean_tfidf').sort_values('mean_tfidf')
#     plt.figure(figsize=(8,5))
#     plt.barh(sub['term'], sub['mean_tfidf'])
#     plt.title(f'Top TF-IDF terms — {label} (cleaned)')
#     plt.xlabel('mean TF-IDF'); plt.tight_layout(); plt.show()

# for lab in sorted(top_df['label'].unique()):
#     plot_top_terms(lab, k=15)


In [34]:
# # gabung 2 kolom jadi satu string
# def combine_and_clean(a, b):
#     a = str(a) if pd.notna(a) else ""
#     b = str(b) if pd.notna(b) else ""
#     combo = f"{a} {b}".strip()
#     return clean_for_analysis(combo)  # pakai fungsi preprocessing yg sudah kita buat

# # buat kolom baru text_bersih
# df['text_bersih'] = [
#     combine_and_clean(a, b) 
#     for a, b in zip(df['text_final'], df['text_nostop'])
# ]

# # cek hasil
# print(df[['text_final','text_nostop','text_bersih']].head(10))

# Embedding + Prediction + Compare

In [35]:
# df.info()

In [36]:
# df

In [37]:
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report

# TEXT_COL  = "text_final"  # hasil preprocessing gabungan
# LABEL_COL = "emotion"

# df_exp = df[[TEXT_COL, LABEL_COL]].dropna().copy()

# X = df_exp[TEXT_COL].astype(str).tolist()
# y_txt = df_exp[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y = le.fit_transform(y_txt)
# print("Classes:", le.classes_)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=42, stratify=y
# )

# # ================================================
# # 2. BASELINE: TF-IDF + CLASSIFIERS
# # ================================================
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
# from xgboost import XGBClassifier

# tfidf = TfidfVectorizer(
#     analyzer="word",
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9,
#     sublinear_tf=True
# )

# X_train_tfidf = tfidf.fit_transform(X_train)
# X_test_tfidf  = tfidf.transform(X_test)

# results = {}

# # Logistic Regression
# logreg = LogisticRegression(max_iter=2000, class_weight="balanced")
# logreg.fit(X_train_tfidf, y_train)
# y_pred = logreg.predict(X_test_tfidf)
# results["TFIDF+LogReg"] = (
#     f1_score(y_test, y_pred, average="macro"),
#     f1_score(y_test, y_pred, average="weighted")
# )

# # Linear SVC
# svc = LinearSVC(class_weight="balanced")
# svc.fit(X_train_tfidf, y_train)
# y_pred = svc.predict(X_test_tfidf)
# results["TFIDF+LinearSVC"] = (
#     f1_score(y_test, y_pred, average="macro"),
#     f1_score(y_test, y_pred, average="weighted")
# )

# # XGBoost
# xgb = XGBClassifier(
#     n_estimators=300, learning_rate=0.1,
#     max_depth=6, subsample=0.8, colsample_bytree=0.8,
#     eval_metric="mlogloss", random_state=42, n_jobs=-1
# )
# xgb.fit(X_train_tfidf, y_train)
# y_pred = xgb.predict(X_test_tfidf)
# results["TFIDF+XGB"] = (
#     f1_score(y_test, y_pred, average="macro"),
#     f1_score(y_test, y_pred, average="weighted")
# )

# # ================================================
# # 3. SENTENCE EMBEDDINGS (distiluse) + LogReg
# # ================================================
# from sentence_transformers import SentenceTransformer
# import numpy as np

# model = SentenceTransformer("distiluse-base-multilingual-cased-v2")

# E_train = model.encode(X_train, batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
# E_test  = model.encode(X_test,  batch_size=64, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

# logreg_emb = LogisticRegression(max_iter=2000, class_weight="balanced")
# logreg_emb.fit(E_train, y_train)
# y_pred = logreg_emb.predict(E_test)
# results["Embeddings+LogReg"] = (
#     f1_score(y_test, y_pred, average="macro"),
#     f1_score(y_test, y_pred, average="weighted")
# )

# # ================================================
# # 4. HASIL
# # ================================================
# import pandas as pd
# res_df = pd.DataFrame([
#     {"Model": k, "F1_macro": v[0], "F1_weighted": v[1]} for k,v in results.items()
# ])
# print(res_df)


In [38]:
# # ================================================
# # TF-IDF baseline + REDUKSI DIMENSI via TruncatedSVD (LSA)
# # ================================================
# from sklearn.decomposition import TruncatedSVD
# from sklearn.preprocessing import Normalizer
# from sklearn.pipeline import make_pipeline
# import numpy as np

# # --- pakai hasil TF-IDF yang sudah kamu buat: X_train_tfidf, X_test_tfidf ---

# results_svd = {}
# svd_dims = [100, 200, 600]   # bisa ubah/expand

# for k in svd_dims:
#     # SVD + Normalizer (umum dipakai di LSA)
#     svd = TruncatedSVD(n_components=k, random_state=42)
#     lsa = make_pipeline(svd, Normalizer(copy=False))

#     X_train_svd = lsa.fit_transform(X_train_tfidf)
#     X_test_svd  = lsa.transform(X_test_tfidf)

#     # LogReg
#     logreg_svd = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs", n_jobs=-1)
#     logreg_svd.fit(X_train_svd, y_train)
#     y_pred = logreg_svd.predict(X_test_svd)
#     results_svd[f"SVD{k}+LogReg"] = (
#         f1_score(y_test, y_pred, average="macro"),
#         f1_score(y_test, y_pred, average="weighted")
#     )

#     # LinearSVC
#     svc_svd = LinearSVC(class_weight="balanced", C=1.0)
#     svc_svd.fit(X_train_svd, y_train)
#     y_pred = svc_svd.predict(X_test_svd)
#     results_svd[f"SVD{k}+LinearSVC"] = (
#         f1_score(y_test, y_pred, average="macro"),
#         f1_score(y_test, y_pred, average="weighted")
#     )

#     # XGBoost (data sudah dense → langsung OK)
#     xgb_svd = XGBClassifier(
#         n_estimators=400, learning_rate=0.1,
#         max_depth=6, subsample=0.9, colsample_bytree=0.9,
#         eval_metric="mlogloss", random_state=42, n_jobs=-1
#     )
#     xgb_svd.fit(X_train_svd, y_train)
#     y_pred = xgb_svd.predict(X_test_svd)
#     results_svd[f"SVD{k}+XGB"] = (
#         f1_score(y_test, y_pred, average="macro"),
#         f1_score(y_test, y_pred, average="weighted")
#     )

# # Gabungkan ringkasan baseline + SVD
# res_all = []
# for k,v in results.items():
#     res_all.append({"Model": k, "F1_macro": v[0], "F1_weighted": v[1]})
# for k,v in results_svd.items():
#     res_all.append({"Model": k, "F1_macro": v[0], "F1_weighted": v[1]})

# res_all_df = pd.DataFrame(res_all).sort_values("F1_macro", ascending=False)
# print(res_all_df)


In [39]:
# import re, numpy as np, pandas as pd, random, warnings
# warnings.filterwarnings("ignore")
# from collections import Counter

# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.calibration import CalibratedClassifierCV
# from sklearn.metrics import f1_score, classification_report, confusion_matrix

# SEED_MAIN = 42
# random.seed(SEED_MAIN); np.random.seed(SEED_MAIN)

# TEXT_COL, LABEL_COL = "text_final", "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns

# def clean_light(s):
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+',' ', s)
#     s = re.sub(r'@\w+',' ', s)
#     s = re.sub(r'[#*_~`]+',' ', s)
#     return re.sub(r'\s+',' ', s).strip()

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# data[TEXT_COL] = data[TEXT_COL].astype(str).apply(clean_light)

# # -------- labels
# le = LabelEncoder()
# y_all = le.fit_transform(data[LABEL_COL].astype(str))
# classes = list(le.classes_)
# X_all = data[TEXT_COL].tolist()

# # -------- split: train / valid / test
# X_train, X_tmp, y_train, y_tmp = train_test_split(
#     X_all, y_all, test_size=0.3, random_state=SEED_MAIN, stratify=y_all
# )
# X_val, X_test, y_val, y_test = train_test_split(
#     X_tmp, y_tmp, test_size=0.5, random_state=SEED_MAIN, stratify=y_tmp
# )

# print("Distribusi Train:", Counter(y_train))
# print("Distribusi Val  :", Counter(y_val))
# print("Distribusi Test :", Counter(y_test))

# # -------- vectorizer (tetap word 1-2, tapi sedikit dirapikan)
# tfidf = TfidfVectorizer(
#     analyzer="word",
#     ngram_range=(1,2),
#     min_df=3,          # sedikit lebih ketat dari 2 (kurangi noise)
#     max_df=0.95,
#     sublinear_tf=True,
#     strip_accents="unicode",
#     max_features=60000 # batasi fitur agar stabil
# )

# Xtr = tfidf.fit_transform(X_train)
# Xva = tfidf.transform(X_val)
# Xte = tfidf.transform(X_test)

# # -------- helper: sample_weight (agresif tapi terkontrol)
# def make_sample_weight(texts, y, alpha=0.7, len_floor=6):
#     """
#     Bobot = (1/freq)^alpha * len_boost
#     - kelas langka → bobot lebih besar
#     - teks sangat pendek → sedikit tambahan bobot
#     """
#     cnt = Counter(y)
#     total = len(y)
#     freq = {c: cnt[c]/total for c in cnt}
#     sw = np.ones(len(y), dtype=np.float32)
#     for i,(t,yi) in enumerate(zip(texts,y)):
#         base = (1.0 / max(freq[yi], 1e-8))**alpha
#         L = max(len(t.split()), 1)
#         len_boost = 1.0 + 0.25*(1.0 if L <= len_floor else 0.0)
#         sw[i] = base * len_boost
#     # normalisasi rata-rata 1 (opsional)
#     sw /= sw.mean()
#     return sw

# sw_train = make_sample_weight(X_train, y_train, alpha=0.7, len_floor=6)

# from collections import Counter

# # --- hitung folds kalibrasi yang aman berdasarkan kelas minor ---
# min_per_class = min(Counter(y_train).values())
# calib_folds = max(2, min(5, min_per_class))  # di [2..5]

# def fit_prob_model(X, y, sw, seed, calib_folds):
#     """
#     Jika tiap kelas punya >= calib_folds sampel -> pakai CalibratedClassifierCV(cv=calib_folds).
#     Jika tidak -> fallback tanpa kalibrasi (pakai prob bawaan LogisticRegression).
#     """
#     base = LogisticRegression(
#         solver="liblinear", penalty="l2",
#         C=2.0, max_iter=3000,
#         class_weight=None,
#         random_state=seed
#     )
#     if calib_folds >= 2:
#         try:
#             from sklearn.calibration import CalibratedClassifierCV
#             calibrated = CalibratedClassifierCV(base_estimator=base, method="sigmoid", cv=calib_folds)
#             calibrated.fit(X, y, sample_weight=sw)
#             # bungkus agar punya .predict_proba/.predict
#             return calibrated
#         except ValueError as e:
#             # fallback jika tetap gagal (mis. distribusi sangat timpang)
#             print(f"[Kalibrasi skip] {e}")
#             base.fit(X, y, sample_weight=sw)
#             return base
#     else:
#         # terlalu sedikit untuk CV kalibrasi
#         base.fit(X, y, sample_weight=sw)
#         return base

# model1 = fit_prob_model(Xtr, y_train, sw_train, seed=SEED_MAIN, calib_folds=calib_folds)
# model2 = fit_prob_model(Xtr, y_train, sw_train, seed=SEED_MAIN+7, calib_folds=calib_folds)

# # -------- threshold search (per-class) di VAL untuk maksimalkan macro-F1
# def tune_thresholds(models, X_val, y_val, grid=np.linspace(0.2, 0.8, 25)):
#     """
#     models: list of calibrated classifiers -> proba dirata-rata
#     kembalikan ambang per kelas (array shape [C])
#     """
#     # rata-rata probabilitas (ensemble)
#     Ps = [m.predict_proba(X_val) for m in models]
#     P = np.mean(Ps, axis=0)  # [N, C]

#     C = P.shape[1]
#     best_t = np.full(C, 1.0/C)  # init
#     # one-vs-rest thresholding: untuk setiap kelas, cari threshold yang maksimalkan F1 klas tersebut,
#     # lalu ulangi koordinat-descent ringan agar macro-F1 keseluruhan naik.
#     # Iterasi kecil (2–3 putaran) sudah cukup.
#     curr_t = best_t.copy()
#     for _ in range(3):
#         for c in range(C):
#             best_f1_c, best_tc = -1, curr_t[c]
#             for t in grid:
#                 pred = P.argmax(axis=1)  # default argmax
#                 # override: jika proba kelas c >= t, pilih c
#                 override = (P[:, c] >= t)
#                 pred[override] = c
#                 f1 = f1_score(y_val, pred, average="macro")
#                 if f1 > best_f1_c:
#                     best_f1_c, best_tc = f1, t
#             curr_t[c] = best_tc
#     return curr_t

# thresholds = tune_thresholds([model1, model2], Xva, y_val)
# print("Ambang per kelas:", {cls: round(th,3) for cls,th in zip(classes, thresholds)})

# # -------- evaluasi di TEST dengan threshold hasil tuning
# def predict_with_thresholds(models, X, thresholds):
#     Ps = [m.predict_proba(X) for m in models]
#     P = np.mean(Ps, axis=0)  # [N,C]
#     pred = P.argmax(axis=1)
#     # override per kelas
#     for c, t in enumerate(thresholds):
#         mask = P[:, c] >= t
#         pred[mask] = c
#     return pred, P

# y_pred, P_test = predict_with_thresholds([model1, model2], Xte, thresholds)

# print("\n=== HASIL TEST (TFIDF+LogReg + sample_weight + threshold tuning + ensemble) ===")
# print("F1_macro   :", round(f1_score(y_test, y_pred, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_test, y_pred, average="weighted"), 6))
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


In [40]:
# # ==============================================
# # Optuna untuk TF-IDF + LogisticRegression
# # fokus: naikkan macro-F1 (5-fold CV), plus test eval
# # ==============================================
# !pip -q install optuna

# import re, numpy as np, pandas as pd, random, warnings, optuna
# from optuna.pruners import MedianPruner
# warnings.filterwarnings("ignore")
# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from scipy.sparse import hstack

# # --------- data ----------
# TEXT_COL, LABEL_COL = "text_final", "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns, "Pastikan df punya text_final & emotion"

# def clean_light(s: str) -> str:
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+',' ', s)
#     s = re.sub(r'@\w+',' ', s)
#     s = re.sub(r'[#*_~`]+',' ', s)
#     return re.sub(r'\s+',' ', s).strip()

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# data[TEXT_COL] = data[TEXT_COL].astype(str).apply(clean_light)

# le = LabelEncoder()
# y_all = le.fit_transform(data[LABEL_COL].astype(str))
# classes = list(le.classes_)
# X_all = data[TEXT_COL].tolist()

# X_tr, X_te, y_tr, y_te = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# # --------- objective ----------
# def objective(trial):
#     # TF-IDF (word)
#     min_df   = trial.suggest_categorical("min_df", [1,2,3,5,7,10])
#     max_df   = trial.suggest_float("max_df", 0.80, 0.98)
#     binary   = trial.suggest_categorical("binary", [False, True])
#     sublin   = trial.suggest_categorical("sublinear_tf", [True, False])
#     stripacc = trial.suggest_categorical("strip_accents", [None, "unicode"])
#     max_feat = trial.suggest_categorical("max_features", [20000, 40000, 60000, 80000, None])
#     ngram_hi = trial.suggest_categorical("word_ngram_high", [3,2])  # (1,2) atau (1,3)

#     vec_w = TfidfVectorizer(
#         analyzer="word", ngram_range=(1, ngram_hi),
#         min_df=min_df, max_df=max_df, binary=binary,
#         sublinear_tf=sublin, strip_accents=stripacc, max_features=max_feat
#     )

#     # Char n-grams (opsional)
#     use_char = trial.suggest_categorical("use_char", [True, False])
#     char_min = trial.suggest_categorical("char_min", [3,4]) if use_char else 3
#     char_max = trial.suggest_categorical("char_max", [5,6]) if use_char else 5
#     char_feat= trial.suggest_categorical("char_max_features", [20000, 40000, None]) if use_char else None
#     char_weight = trial.suggest_float("char_weight", 0.3, 1.5) if use_char else 1.0

#     # LogReg hyperparams
#     solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
#     penalty = "l2" if solver=="lbfgs" else trial.suggest_categorical("penalty", ["l1","l2"])
#     C      = trial.suggest_float("C", 1e-3, 3e1, log=True)
#     cw_opt = trial.suggest_categorical("class_weight", [None, "balanced"])

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
#     f1s = []

#     for tr_idx, va_idx in skf.split(X_tr, y_tr):
#         Xtr_text = [X_tr[i] for i in tr_idx]; Xva_text = [X_tr[i] for i in va_idx]
#         ytr = y_tr[tr_idx]; yva = y_tr[va_idx]

#         Xtr_w = vec_w.fit_transform(Xtr_text)
#         Xva_w = vec_w.transform(Xva_text)

#         if use_char:
#             vec_c = TfidfVectorizer(
#                 analyzer="char", ngram_range=(char_min, char_max),
#                 min_df=2, sublinear_tf=True, max_features=char_feat
#             )
#             Xtr_c = vec_c.fit_transform(Xtr_text)
#             Xva_c = vec_c.transform(Xva_text)

#             # skala blok char agar kontribusinya bisa dikontrol
#             if char_weight != 1.0:
#                 Xtr_c = Xtr_c * char_weight
#                 Xva_c = Xva_c * char_weight

#             Xtr = hstack([Xtr_w, Xtr_c], format="csr")
#             Xva = hstack([Xva_w, Xva_c], format="csr")
#         else:
#             Xtr, Xva = Xtr_w, Xva_w

#         clf = LogisticRegression(
#             max_iter=3000, solver=solver, penalty=penalty, C=C,
#             class_weight=cw_opt, n_jobs=-1 if solver!="liblinear" else None
#         )
#         clf.fit(Xtr, ytr)
#         pred = clf.predict(Xva)
#         f1s.append(f1_score(yva, pred, average="macro"))

#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=80, show_progress_bar=True)  # bisa dinaikkan 120–200

# print("\nBest CV macro-F1:", round(study.best_value, 6))
# bp = study.best_params
# print("Best params:", bp)

# # --------- Refit best di TRAIN dan evaluasi di TEST ----------
# vec_w = TfidfVectorizer(
#     analyzer="word", ngram_range=(1, bp["word_ngram_high"]),
#     min_df=bp["min_df"], max_df=bp["max_df"], binary=bp["binary"],
#     sublinear_tf=bp["sublinear_tf"], strip_accents=bp["strip_accents"],
#     max_features=bp["max_features"]
# )
# Xtr_w = vec_w.fit_transform(X_tr)
# Xte_w = vec_w.transform(X_te)

# use_char = bp["use_char"]
# if use_char:
#     vec_c = TfidfVectorizer(
#         analyzer="char",
#         ngram_range=(bp["char_min"], bp["char_max"]),
#         min_df=2, sublinear_tf=True,
#         max_features=bp["char_max_features"]
#     )
#     Xtr_c = vec_c.fit_transform(X_tr)
#     Xte_c = vec_c.transform(X_te)
#     if bp["char_weight"] != 1.0:
#         Xtr_c = Xtr_c * bp["char_weight"]
#         Xte_c = Xte_c * bp["char_weight"]
#     Xtr_all = hstack([Xtr_w, Xtr_c], format="csr")
#     Xte_all = hstack([Xte_w, Xte_c], format="csr")
# else:
#     Xtr_all, Xte_all = Xtr_w, Xte_w

# solver  = bp["solver"]
# penalty = "l2" if solver=="lbfgs" else bp["penalty"]
# clf = LogisticRegression(
#     max_iter=3000, solver=solver, penalty=penalty, C=bp["C"],
#     class_weight=bp["class_weight"], n_jobs=-1 if solver!="liblinear" else None
# )
# clf.fit(Xtr_all, y_tr)
# yhat = clf.predict(Xte_all)

# from sklearn.metrics import f1_score
# print("\n=== TEST (Optuna-best TFIDF + LogReg) ===")
# print("F1_macro   :", round(f1_score(y_te, yhat, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_te, yhat, average="weighted"), 6))

# print("\nClassification Report:")
# print(classification_report(y_te, yhat, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_te, yhat))


In [41]:
# # ==============================================
# # TF-IDF (1,2) [+ optional char(3,5)] 
# # -> L1 Logistic (feature selection)
# # -> L2 Logistic (final)
# # Tujuan: naikkan macro-F1 tanpa SVD
# # ==============================================
# !pip -q install optuna

# import re, numpy as np, pandas as pd, random, warnings, optuna
# from optuna.pruners import MedianPruner
# warnings.filterwarnings("ignore")
# random.seed(42); np.random.seed(42)

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from scipy.sparse import hstack, csr_matrix
# from sklearn.linear_model import LogisticRegression

# # ---------- data ----------
# TEXT_COL, LABEL_COL = "text_final", "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns

# def clean_light(s: str) -> str:
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+',' ', s)
#     s = re.sub(r'@\w+',' ', s)
#     s = re.sub(r'[#*_~`]+',' ', s)
#     return re.sub(r'\s+',' ', s).strip()

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# data[TEXT_COL] = data[TEXT_COL].astype(str).apply(clean_light)

# le = LabelEncoder()
# y_all = le.fit_transform(data[LABEL_COL].astype(str))
# classes = list(le.classes_)
# X_all = data[TEXT_COL].tolist()

# X_tr, X_te, y_tr, y_te = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
# )

# # ---------- objective ----------
# def objective(trial):
#     # TF-IDF word
#     min_df   = trial.suggest_categorical("min_df", [1,2,3,5])
#     max_df   = trial.suggest_float("max_df", 0.8, 0.98)
#     sublin   = trial.suggest_categorical("sublinear_tf", [True, False])
#     binary   = trial.suggest_categorical("binary", [False, True])
#     max_feat = trial.suggest_categorical("max_features", [20000, 40000, 60000, None])
#     stripacc = trial.suggest_categorical("strip_accents", [None, "unicode"])
#     use_char = trial.suggest_categorical("use_char", [True, False])

#     vec_w = TfidfVectorizer(
#         analyzer="word", ngram_range=(1,2),
#         min_df=min_df, max_df=max_df,
#         sublinear_tf=sublin, binary=binary,
#         max_features=max_feat, strip_accents=stripacc
#     )
#     Xw = vec_w.fit_transform(X_tr)

#     if use_char:
#         vec_c = TfidfVectorizer(
#             analyzer="char", ngram_range=(3,5),
#             min_df=2, sublinear_tf=True, binary=False,
#             max_features=30000
#         )
#         Xc = vec_c.fit_transform(X_tr)
#         Xs = hstack([Xw, Xc], format="csr")
#     else:
#         Xs = Xw

#     # Tahap 1: L1 untuk feature selection
#     C1 = trial.suggest_float("C1_l1", 1e-3, 3.0, log=True)
#     l1 = LogisticRegression(
#         penalty="l1", solver="liblinear", max_iter=3000,
#         class_weight="balanced", C=C1
#     )

#     # Tahap 2: L2 untuk final
#     C2 = trial.suggest_float("C2_l2", 1e-3, 10.0, log=True)
#     l2 = LogisticRegression(
#         penalty="l2", solver="lbfgs", max_iter=3000, n_jobs=-1,
#         class_weight="balanced", C=C2
#     )

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     f1s = []
#     for tr_idx, va_idx in skf.split(Xs, y_tr):
#         X_tr_f, X_va_f = Xs[tr_idx], Xs[va_idx]
#         y_tr_f, y_va_f = y_tr[tr_idx], y_tr[va_idx]

#         # fit L1 -> ambil kolom non-zero (per kelas, union)
#         l1.fit(X_tr_f, y_tr_f)
#         coef = l1.coef_
#         keep = np.unique(np.where(coef!=0)[1]) if coef.ndim==2 else np.where(coef!=0)[0]
#         if keep.size == 0:
#             # fallback kalau semua nol
#             keep = np.arange(X_tr_f.shape[1])[: min(1000, X_tr_f.shape[1])]

#         X_tr_sel = X_tr_f[:, keep]
#         X_va_sel = X_va_f[:, keep]

#         l2.fit(X_tr_sel, y_tr_f)
#         y_hat = l2.predict(X_va_sel)
#         f1s.append(f1_score(y_va_f, y_hat, average="macro"))
#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=60, show_progress_bar=True)

# bp = study.best_params
# print("\nBest CV macro-F1:", round(study.best_value, 6))
# print("Best params:", bp)

# # ---------- train final di full-train, eval test ----------
# # Build vectorizers sesuai best
# vec_w = TfidfVectorizer(
#     analyzer="word", ngram_range=(1,2),
#     min_df=bp["min_df"], max_df=bp["max_df"],
#     sublinear_tf=bp["sublinear_tf"], binary=bp["binary"],
#     max_features=bp["max_features"], strip_accents=bp["strip_accents"]
# )
# Xw_tr = vec_w.fit_transform(X_tr)
# Xw_te = vec_w.transform(X_te)

# if bp["use_char"]:
#     from sklearn.feature_extraction.text import TfidfVectorizer
#     vec_c = TfidfVectorizer(
#         analyzer="char", ngram_range=(3,5),
#         min_df=2, sublinear_tf=True, binary=False,
#         max_features=30000
#     )
#     Xc_tr = vec_c.fit_transform(X_tr)
#     Xc_te = vec_c.transform(X_te)
#     X_tr_all = hstack([Xw_tr, Xc_tr], format="csr")
#     X_te_all = hstack([Xw_te, Xc_te], format="csr")
# else:
#     X_tr_all, X_te_all = Xw_tr, Xw_te

# # Tahap 1: L1 (feature selection) pakai full-train
# l1 = LogisticRegression(
#     penalty="l1", solver="liblinear", max_iter=3000,
#     class_weight="balanced", C=bp["C1_l1"]
# )
# l1.fit(X_tr_all, y_tr)
# coef = l1.coef_
# keep = np.unique(np.where(coef!=0)[1]) if coef.ndim==2 else np.where(coef!=0)[0]
# if keep.size == 0:
#     keep = np.arange(X_tr_all.shape[1])[: min(2000, X_tr_all.shape[1])]
# X_tr_sel = X_tr_all[:, keep]
# X_te_sel = X_te_all[:, keep]
# print("Selected features:", X_tr_sel.shape[1], "dari", X_tr_all.shape[1])

# # Tahap 2: L2 final
# l2 = LogisticRegression(
#     penalty="l2", solver="lbfgs", max_iter=3000, n_jobs=-1,
#     class_weight="balanced", C=bp["C2_l2"]
# )
# l2.fit(X_tr_sel, y_tr)
# y_hat = l2.predict(X_te_sel)

# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# print("\n=== TEST (TF-IDF -> L1 select -> L2 final) ===")
# print("F1_macro   :", round(f1_score(y_te, y_hat, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_te, y_hat, average="weighted"), 6))
# print("\nClassification Report:")
# print(classification_report(y_te, y_hat, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_te, y_hat))


In [42]:
# # ============================================
# # BOOST TF-IDF BASELINE:
# # (A) Word(1,2)+Char(3,5) + Chi2 feature selection + LinearSVC
# # (B) NB-SVM (log-count ratio) + LinearSVC
# # (C) Soft-Voting Ensemble: LogReg + Calibrated LinearSVC
# # ============================================
# import re, numpy as np, pandas as pd, warnings, random
# warnings.filterwarnings("ignore")

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder, normalize
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.feature_selection import SelectKBest, chi2
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
# from sklearn.calibration import CalibratedClassifierCV
# from scipy.sparse import hstack, csr_matrix

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# TEXT_COL, LABEL_COL = "text_final", "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns

# def clean_light(s):
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+',' ', s)
#     s = re.sub(r'@\w+',' ', s)
#     s = re.sub(r'[#*_~`]+',' ', s)
#     return re.sub(r'\s+',' ', s).strip()

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X_all = data[TEXT_COL].astype(str).apply(clean_light).tolist()
# y_txt = data[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# classes = list(le.classes_)
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# results = {}

# # ---------------------------
# # Baseline kamu (untuk referensi)
# # ---------------------------
# vec_base = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_df=0.9, sublinear_tf=True)
# Xtr_b = vec_base.fit_transform(X_train); Xte_b = vec_base.transform(X_test)
# logreg_b = LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs", n_jobs=-1)
# logreg_b.fit(Xtr_b, y_train)
# pred_b = logreg_b.predict(Xte_b)
# results["Baseline TFIDF(1,2)+LogReg"] = (
#     f1_score(y_test, pred_b, average="macro"),
#     f1_score(y_test, pred_b, average="weighted")
# )

# # ==========================================================
# # (A) Word(1,2) + Char(3,5) + Chi2 selection + LinearSVC
# # ==========================================================
# vec_word = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=2, max_df=0.95, sublinear_tf=True, max_features=40000)
# vec_char = TfidfVectorizer(analyzer="char", ngram_range=(3,5), min_df=2, sublinear_tf=True, max_features=30000)

# Xtr_w = vec_word.fit_transform(X_train); Xte_w = vec_word.transform(X_test)
# Xtr_c = vec_char.fit_transform(X_train); Xte_c = vec_char.transform(X_test)

# Xtr_wc = hstack([Xtr_w, Xtr_c], format="csr")
# Xte_wc = hstack([Xte_w, Xte_c], format="csr")

# # Chi2 feature selection (coba ambil 60% fitur)
# k = int(Xtr_wc.shape[1] * 0.6)
# selector = SelectKBest(chi2, k=k).fit(Xtr_wc, y_train)
# Xtr_wc_sel = selector.transform(Xtr_wc)
# Xte_wc_sel = selector.transform(Xte_wc)

# svcA = LinearSVC(C=2.0, class_weight="balanced")  # C bisa di-grid [0.5,1,2,3]
# svcA.fit(Xtr_wc_sel, y_train)
# predA = svcA.predict(Xte_wc_sel)
# results["A: word+char+chi2 + LinearSVC"] = (
#     f1_score(y_test, predA, average="macro"),
#     f1_score(y_test, predA, average="weighted")
# )

# # ==========================================
# # (B) NB-SVM: log-count ratio + LinearSVC
# #    (Wang & Manning 2012, very strong for text)
# # ==========================================
# # 1) vectorizer unigram+bigram binary counts
# vec_nb = TfidfVectorizer(analyzer="word", ngram_range=(1,2), min_df=1, max_df=0.99, binary=True, use_idf=False, norm=None)
# Xtr_nb = vec_nb.fit_transform(X_train); Xte_nb = vec_nb.transform(X_test)

# # 2) hitung log-count ratio r = log( (p(w|y=1)+α)/(p(w|y=0)+α) ) untuk multiclass -> one-vs-rest
# #    kita transform fitur per kelas saat training via wrapper
# alpha = 1.0

# def log_count_ratio(X, y, c, alpha=1.0):
#     # X: csr, y: array, c: kelas target (ovr)
#     # p(w|c) dan p(w|~c)
#     pos = (y == c).astype(np.int32)
#     neg = (y != c).astype(np.int32)
#     pw = X[pos==1].sum(axis=0) + alpha
#     qw = X[neg==1].sum(axis=0) + alpha
#     r = np.log((pw / pw.sum()) / (qw / qw.sum()))
#     return np.asarray(r).ravel()  # shape (V,)

# class NBSVM_OVR:
#     def __init__(self, C=2.0):
#         self.C = C
#         self.rs = None
#         self.clfs = []
#     def fit(self, X, y):
#         n_classes = len(np.unique(y))
#         self.rs = []
#         self.clfs = []
#         for c in range(n_classes):
#             r = log_count_ratio(X, y, c, alpha=alpha)
#             self.rs.append(r)
#             Xr = X.multiply(r)  # NB transform
#             clf = LinearSVC(C=self.C, class_weight="balanced")
#             clf.fit(Xr, (y==c).astype(int))
#             self.clfs.append(clf)
#         return self
#     def predict(self, X):
#         # skor = distance to hyperplane untuk tiap kelas
#         scores = []
#         for clf, r in zip(self.clfs, self.rs):
#             s = clf.decision_function(X.multiply(r))
#             scores.append(s.reshape(-1,1))
#         S = np.hstack(scores)
#         return np.argmax(S, axis=1)

# nbsvm = NBSVM_OVR(C=2.0)
# nbsvm.fit(Xtr_nb, y_train)
# predB = nbsvm.predict(Xte_nb)
# results["B: NB-SVM (NBSVM-OVR)"] = (
#     f1_score(y_test, predB, average="macro"),
#     f1_score(y_test, predB, average="weighted")
# )

# # ==========================================
# # (C) Soft-voting: LogReg + Calibrated LinearSVC
# # ==========================================
# # LogReg prob
# logreg = LogisticRegression(max_iter=3000, class_weight="balanced", solver="lbfgs", n_jobs=-1)
# logreg.fit(Xtr_b, y_train)

# # LinearSVC tidak punya prob -> kalibrasi (sigmoid)
# svc_cal = CalibratedClassifierCV(
#     base_estimator=LinearSVC(C=1.0, class_weight="balanced"),
#     method="sigmoid",
#     cv=5
# )
# svc_cal.fit(Xtr_b, y_train)

# P_lr = logreg.predict_proba(Xte_b)
# P_svc = svc_cal.predict_proba(Xte_b)
# # rata-rata probabilitas (bisa beri bobot, mis. 0.6*svc + 0.4*lr)
# P_ens = 0.5 * P_lr + 0.5 * P_svc
# predC = P_ens.argmax(axis=1)
# results["C: Ensemble(LogReg + Calibrated SVC)"] = (
#     f1_score(y_test, predC, average="macro"),
#     f1_score(y_test, predC, average="weighted")
# )

# # ---------------------------
# # RINGKASAN
# # ---------------------------
# res_df = pd.DataFrame(
#     [{"Model": k, "F1_macro": v[0], "F1_weighted": v[1]} for k,v in results.items()]
# ).sort_values("F1_macro", ascending=False)
# print(res_df)

# # (opsional) tampilkan klasifikasi report model terbaik
# best_name = res_df.iloc[0]["Model"]
# best_pred = {"Baseline TFIDF(1,2)+LogReg": pred_b,
#              "A: word+char+chi2 + LinearSVC": predA,
#              "B: NB-SVM (NBSVM-OVR)": predB,
#              "C: Ensemble(LogReg + Calibrated SVC)": predC}[best_name]

# print(f"\n=== BEST: {best_name} ===")
# print("Macro-F1:", round(f1_score(y_test, best_pred, average='macro'), 4))
# print("Weighted-F1:", round(f1_score(y_test, best_pred, average='weighted'), 4))
# print("\nClassification Report:")
# print(classification_report(y_test, best_pred, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_test, best_pred))


In [43]:
# # ==============================================
# # Baseline++: TF-IDF (1,2) + LogisticRegression
# #   (1) Optuna tuning
# #   (2) Targeted reweighting (berdasarkan F1 per kelas)
# # ==============================================
# !pip -q install optuna

# import re, numpy as np, pandas as pd, random, warnings, optuna
# from optuna.pruners import MedianPruner
# warnings.filterwarnings("ignore")
# random.seed(42); np.random.seed(42)

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score, classification_report, confusion_matrix

# TEXT_COL, LABEL_COL = "text_final", "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns

# def clean_light(s: str) -> str:
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
#     s = re.sub(r'@\w+', ' ', s)
#     s = re.sub(r'[#*_~`]', ' ', s)
#     return re.sub(r'\s+', ' ', s).strip()

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# data[TEXT_COL] = data[TEXT_COL].astype(str).apply(clean_light)

# le = LabelEncoder()
# y_all = le.fit_transform(data[LABEL_COL].astype(str))
# classes = list(le.classes_)
# X_all = data[TEXT_COL].tolist()

# X_tr, X_te, y_tr, y_te = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
# )

# # ------------------------------
# # (1) OPTUNA TUNING
# # ------------------------------
# def objective(trial):
#     # Vectorizer search (masih word (1,2) karena itu baseline terbaik)
#     min_df   = trial.suggest_categorical("min_df", [1,2,3,5])
#     max_df   = trial.suggest_float("max_df", 0.75, 0.98)
#     sublin   = trial.suggest_categorical("sublinear_tf", [True, False])
#     max_feat = trial.suggest_categorical("max_features", [20000, 40000, 60000, None])
#     stripacc = trial.suggest_categorical("strip_accents", [None, "unicode"])
#     use_binary = trial.suggest_categorical("binary", [False, True])
#     norm_opt = trial.suggest_categorical("norm", ["l2", None])  # kadang None works

#     vec = TfidfVectorizer(
#         analyzer="word", ngram_range=(1,2),
#         min_df=min_df, max_df=max_df,
#         sublinear_tf=sublin, max_features=max_feat,
#         strip_accents=stripacc, binary=use_binary, norm=norm_opt
#     )

#     # LogReg search
#     solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
#     C      = trial.suggest_float("C", 1e-3, 3e1, log=True)
#     # liblinear boleh L1/L2, lbfgs hanya L2
#     penalty = "l2" if solver=="lbfgs" else trial.suggest_categorical("penalty", ["l1","l2"])
#     # class weight baseline
#     cw_opt = trial.suggest_categorical("class_weight", [None, "balanced"])

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#     f1s = []
#     for tr_idx, va_idx in skf.split(X_tr, y_tr):
#         Xt_tr = [X_tr[i] for i in tr_idx]; y_tr_f = y_tr[tr_idx]
#         Xt_va = [X_tr[i] for i in va_idx]; y_va_f = y_tr[va_idx]

#         Xtr = vec.fit_transform(Xt_tr)
#         Xva = vec.transform(Xt_va)

#         clf = LogisticRegression(
#             max_iter=3000,
#             solver=solver,
#             penalty=penalty,
#             C=C,
#             class_weight=cw_opt,
#             n_jobs=-1 if solver!="liblinear" else None
#         )
#         clf.fit(Xtr, y_tr_f)
#         yhat = clf.predict(Xva)
#         f1s.append(f1_score(y_va_f, yhat, average="macro"))
#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=60, show_progress_bar=True)

# bp = study.best_params
# print("\nBest CV macro-F1:", round(study.best_value, 6))
# print("Best params:", bp)

# # Refit dengan best params
# vec_best = TfidfVectorizer(
#     analyzer="word", ngram_range=(1,1),
#     min_df=bp["min_df"], max_df=bp["max_df"],
#     sublinear_tf=bp["sublinear_tf"], max_features=bp["max_features"],
#     strip_accents=bp["strip_accents"], binary=bp["binary"], norm=bp["norm"]
# )
# Xtr = vec_best.fit_transform(X_tr)
# Xte = vec_best.transform(X_te)

# solver  = bp["solver"]
# penalty = "l2" if solver=="lbfgs" else bp["penalty"]
# cw_base = bp["class_weight"]

# clf_base = LogisticRegression(
#     max_iter=3000, solver=solver, penalty=penalty, C=bp["C"],
#     class_weight=cw_base, n_jobs=-1 if solver!="liblinear" else None
# )
# clf_base.fit(Xtr, y_tr)
# yhat_base = clf_base.predict(Xte)

# print("\n=== TEST (Optuna-best) ===")
# print("F1_macro   :", round(f1_score(y_te, yhat_base, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_te, yhat_base, average="weighted"), 6))
# print(classification_report(y_te, yhat_base, target_names=classes, digits=3))

# # ------------------------------
# # (2) TARGETED REWEIGHTING (2 putaran)
# # ------------------------------
# def targeted_weights_from_val(y_true, y_pred, power_freq=0.5, power_f1=1.0):
#     # bobot kelas = (1/freq)^alpha * (1/f1)^beta
#     from collections import Counter
#     cnt = Counter(y_true)
#     f1_per_class = []
#     for c in range(len(classes)):
#         # f1 kelas c
#         y_true_c = (y_true==c).astype(int)
#         y_pred_c = (y_pred==c).astype(int)
#         tp = np.sum((y_true_c==1) & (y_pred_c==1))
#         fp = np.sum((y_true_c==0) & (y_pred_c==1))
#         fn = np.sum((y_true_c==1) & (y_pred_c==0))
#         prec = tp / (tp+fp+1e-8)
#         rec  = tp / (tp+fn+1e-8)
#         f1c  = 2*prec*rec/(prec+rec+1e-8)
#         f1_per_class.append(max(f1c, 1e-6))
#     total = sum(cnt.values())
#     weights = {}
#     for c in range(len(classes)):
#         freq = cnt.get(c,0)/total if total>0 else 1e-6
#         w = ( (1.0/max(freq,1e-6))**power_freq ) * ( (1.0/f1_per_class[c])**power_f1 )
#         weights[c] = float(w)
#     # normalisasi biar rata2=1 (opsional)
#     meanw = np.mean(list(weights.values()))
#     weights = {k:v/meanw for k,v in weights.items()}
#     return weights

# # bagi train lagi jadi train/val kecil untuk dapat F1 per kelas
# X_tr_in, X_val_in, y_tr_in, y_val_in = train_test_split(
#     X_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr
# )
# Xtr_in = vec_best.fit_transform(X_tr_in); Xval_in = vec_best.transform(X_val_in)

# def fit_with_weights(weights_dict):
#     # map ke dict kelas→bobot
#     cw = {i:weights_dict[i] for i in range(len(classes))}
#     clf = LogisticRegression(
#         max_iter=3000, solver=solver, penalty=penalty, C=bp["C"],
#         class_weight=cw, n_jobs=-1 if solver!="liblinear" else None
#     )
#     clf.fit(Xtr_in, y_tr_in)
#     return clf

# # putaran 1: pakai hasil prediksi awal (optuna-best) sebagai proxy, lalu hitung bobot
# clf_tmp = LogisticRegression(
#     max_iter=3000, solver=solver, penalty=penalty, C=bp["C"],
#     class_weight=cw_base, n_jobs=-1 if solver!="liblinear" else None
# )
# clf_tmp.fit(Xtr_in, y_tr_in)
# pred_in = clf_tmp.predict(Xval_in)
# w1 = targeted_weights_from_val(y_val_in, pred_in, power_freq=0.5, power_f1=1.0)

# # putaran 2: fit ulang di (train_in) pakai w1, hitung bobot baru w2
# clf_w1 = fit_with_weights(w1)
# pred_in2 = clf_w1.predict(Xval_in)
# w2 = targeted_weights_from_val(y_val_in, pred_in2, power_freq=0.5, power_f1=1.0)

# # final: refit di full TRAIN (X_tr, y_tr) pakai bobot w2 -> evaluasi TEST
# Xtr_full = vec_best.fit_transform(X_tr)
# clf_final = LogisticRegression(
#     max_iter=3000, solver=solver, penalty=penalty, C=bp["C"],
#     class_weight={i:w2[i] for i in range(len(classes))},
#     n_jobs=-1 if solver!="liblinear" else None
# )
# clf_final.fit(Xtr_full, y_tr)
# yhat_final = clf_final.predict(Xte)

# print("\n=== TEST (Baseline++ dengan Targeted Reweighting) ===")
# print("F1_macro   :", round(f1_score(y_te, yhat_final, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_te, yhat_final, average="weighted"), 6))
# print(classification_report(y_te, yhat_final, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_te, yhat_final))

In [44]:
# # ===========================================
# # TF-IDF (UNIGRAM ONLY) + Logistic Regression
# # Hyperparameter tuning with Optuna (maximize F1_macro)
# # ===========================================
# !pip -q install optuna

# import re, numpy as np, pandas as pd, warnings, random, optuna
# from optuna.pruners import MedianPruner
# warnings.filterwarnings("ignore")

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score, classification_report, confusion_matrix

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# # ---------- config ----------
# TEXT_COL  = "text_final"           # pakai text_final, tanpa stemming
# LABEL_COL = "emotion"              # sesuai kode kamu
# TEST_SIZE = 0.2
# N_SPLITS  = 5
# N_TRIALS  = 60                     # bisa naikkan ke 100-200 kalau mau

# assert TEXT_COL in df.columns and LABEL_COL in df.columns, "Pastikan df punya text_final & emotion."

# # ringan: bersihkan URL/simbol tanpa stemming
# def clean_light(s: str) -> str:
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+', ' ', s)   # URL
#     s = re.sub(r'@\w+', ' ', s)                    # mention
#     s = re.sub(r'[#*_~`]', ' ', s)                 # simbol ringan
#     s = re.sub(r'\s+', ' ', s).strip()
#     return s

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X_all = data[TEXT_COL].astype(str).apply(clean_light).tolist()
# y_txt = data[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# classes = list(le.classes_)
# print("Classes:", classes)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=TEST_SIZE, random_state=SEED, stratify=y_all
# )

# def objective(trial: optuna.Trial) -> float:
#     # --- TF-IDF UNIGRAM ONLY ---
#     min_df   = trial.suggest_categorical("min_df", [1, 2, 3, 5])
#     max_df   = trial.suggest_float("max_df", 0.6, 0.99)
#     sublin   = trial.suggest_categorical("sublinear_tf", [True, False])
#     max_feat = trial.suggest_categorical("max_features", [20000, 30000, 50000, None])

#     vec = TfidfVectorizer(
#         analyzer="word",
#         ngram_range=(1,1),            # <— UNIGRAM ONLY
#         min_df=min_df,
#         max_df=max_df,
#         sublinear_tf=sublin,
#         max_features=max_feat
#     )

#     # --- Logistic Regression params ---
#     solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
#     C      = trial.suggest_float("C", 1e-4, 1e4, log=True)
#     cweight= trial.suggest_categorical("class_weight", [None, "balanced"])
#     # liblinear boleh L1/L2; lbfgs hanya L2
#     if solver == "liblinear":
#         penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
#     else:
#         penalty = "l2"

#     skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
#     f1s = []

#     for tr_idx, va_idx in skf.split(X_train, y_train):
#         X_tr_text = [X_train[i] for i in tr_idx]
#         X_va_text = [X_train[i] for i in va_idx]
#         y_tr = y_train[tr_idx]
#         y_va = y_train[va_idx]

#         Xtr = vec.fit_transform(X_tr_text)
#         Xva = vec.transform(X_va_text)

#         clf = LogisticRegression(
#             max_iter=3000,
#             class_weight=cweight,
#             C=C,
#             solver=solver,
#             penalty=penalty,
#             n_jobs=-1 if solver != "liblinear" else None,
#         )
#         clf.fit(Xtr, y_tr)
#         y_hat = clf.predict(Xva)
#         f1s.append(f1_score(y_va, y_hat, average="macro"))

#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# print("\nBest trial #", study.best_trial.number)
# print("Best CV macro-F1:", round(study.best_value, 6))
# print("Best params:", study.best_params)

# # ---------- Refit best on full train → evaluate on test ----------
# bp = study.best_params

# vec_best = TfidfVectorizer(
#     analyzer="word",
#     ngram_range=(1,1),                         # UNIGRAM ONLY
#     min_df=bp["min_df"],
#     max_df=bp["max_df"],
#     sublinear_tf=bp["sublinear_tf"],
#     max_features=bp["max_features"]
# )
# Xtr = vec_best.fit_transform(X_train)
# Xte = vec_best.transform(X_test)

# solver = bp["solver"]
# penalty = bp.get("penalty", "l2") if solver == "liblinear" else "l2"

# clf_best = LogisticRegression(
#     max_iter=3000,
#     class_weight=bp["class_weight"],
#     C=bp["C"],
#     solver=solver,
#     penalty=penalty,
#     n_jobs=-1 if solver != "liblinear" else None,
# )
# clf_best.fit(Xtr, y_train)
# y_pred = clf_best.predict(Xte)

# print("\n=== TEST (Optuna-best TF-IDF unigram + LogReg) ===")
# print("F1_macro   :", round(f1_score(y_test, y_pred, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_test, y_pred, average="weighted"), 6))
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


In [45]:
# # ===========================================
# # TF-IDF BIGRAM + Logistic Regression (Optuna)
# # ===========================================
# !pip -q install optuna

# import re, numpy as np, pandas as pd, warnings, random, optuna
# from optuna.pruners import MedianPruner
# warnings.filterwarnings("ignore")

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import f1_score, classification_report, confusion_matrix

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# # --------- CONFIG ---------
# TEXT_COL  = "text_final"
# LABEL_COL = "emotion"
# TEST_SIZE = 0.2
# N_SPLITS  = 5
# N_TRIALS  = 60

# # set True bila ingin uni+bi (1,2); False = bigram-only (2,2)
# USE_UNIGRAM_TOO = True

# assert TEXT_COL in df.columns and LABEL_COL in df.columns, "Pastikan df punya text_final & emotion."

# def clean_light(s: str) -> str:
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
#     s = re.sub(r'@\w+', ' ', s)
#     s = re.sub(r'[#*_~`]', ' ', s)
#     s = re.sub(r'\s+', ' ', s).strip()
#     return s

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X_all = data[TEXT_COL].astype(str).apply(clean_light).tolist()
# y_txt = data[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# classes = list(le.classes_)
# print("Classes:", classes)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=TEST_SIZE, random_state=SEED, stratify=y_all
# )

# def objective(trial: optuna.Trial) -> float:
#     # -------- TF-IDF BIGRAM --------
#     min_df   = trial.suggest_categorical("min_df", [1, 2, 3, 5])
#     max_df   = trial.suggest_float("max_df", 0.7, 0.98)
#     sublin   = trial.suggest_categorical("sublinear_tf", [True, False])
#     max_feat = trial.suggest_categorical("max_features", [20000, 30000, 50000, None])

#     ngram_low = 1 if USE_UNIGRAM_TOO else 2
#     ngram_hi  = 2

#     vec = TfidfVectorizer(
#         analyzer="word",
#         ngram_range=(ngram_low, ngram_hi),   # (2,2) bigram-only; (1,2) uni+bi
#         min_df=min_df,
#         max_df=max_df,
#         sublinear_tf=sublin,
#         max_features=max_feat
#     )

#     # -------- Logistic Regression --------
#     solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
#     C      = trial.suggest_float("C", 1e-3, 1e2, log=True)
#     cweight= trial.suggest_categorical("class_weight", [None, "balanced"])
#     penalty = "l2" if solver=="lbfgs" else trial.suggest_categorical("penalty", ["l1","l2"])

#     skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
#     f1s = []
#     for tr_idx, va_idx in skf.split(X_train, y_train):
#         Xt_tr = [X_train[i] for i in tr_idx]
#         Xt_va = [X_train[i] for i in va_idx]
#         y_tr, y_va = y_train[tr_idx], y_train[va_idx]

#         Xtr = vec.fit_transform(Xt_tr)
#         Xva = vec.transform(Xt_va)

#         clf = LogisticRegression(
#             max_iter=3000,
#             class_weight=cweight,
#             C=C,
#             solver=solver,
#             penalty=penalty,
#             n_jobs=-1 if solver!="liblinear" else None
#         )
#         clf.fit(Xtr, y_tr)
#         y_hat = clf.predict(Xva)
#         f1s.append(f1_score(y_va, y_hat, average="macro"))

#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# print("\nBest trial #", study.best_trial.number)
# print("Best CV macro-F1:", round(study.best_value, 6))
# print("Best params:", study.best_params)

# # -------- Refit best on full train -> test --------
# bp = study.best_params
# vec_best = TfidfVectorizer(
#     analyzer="word",
#     ngram_range=((1 if USE_UNIGRAM_TOO else 2), 2),
#     min_df=bp["min_df"],
#     max_df=bp["max_df"],
#     sublinear_tf=bp["sublinear_tf"],
#     max_features=bp["max_features"]
# )

# Xtr = vec_best.fit_transform(X_train)
# Xte = vec_best.transform(X_test)

# solver  = bp["solver"]
# penalty = "l2" if solver=="lbfgs" else bp["penalty"]

# clf_best = LogisticRegression(
#     max_iter=3000,
#     class_weight=bp["class_weight"],
#     C=bp["C"],
#     solver=solver,
#     penalty=penalty,
#     n_jobs=-1 if solver!="liblinear" else None
# )
# clf_best.fit(Xtr, y_train)
# y_pred = clf_best.predict(Xte)

# print("\n=== TEST (Optuna-best TF-IDF BIGRAM + LogReg) ===")
# print("F1_macro   :", round(f1_score(y_test, y_pred, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_test, y_pred, average="weighted"), 6))
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


In [46]:
# # ============================================================
# # HYBRID: TF-IDF (bigram + char) + DistilUSE embeddings
# # Classifier: LinearSVC(class_weight="balanced")
# # Oversampling: train-only (manual) untuk kelas minor
# # Target: naikkan F1_macro >= 0.30
# # ============================================================
# !pip -q install sentence-transformers

# import os, re, random, warnings, numpy as np, pandas as pd
# warnings.filterwarnings("ignore")
# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder, normalize
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.svm import LinearSVC
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from scipy.sparse import hstack, csr_matrix
# from sentence_transformers import SentenceTransformer
# from collections import Counter

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# # -------------------
# # KONFIG
# # -------------------
# TEXT_COL  = "text_final"          # pakai text_final (tanpa stemming)
# LABEL_COL = "emotion"             # ganti jika labelmu beda
# USE_CHAR  = True                  # tambahkan char n-gram (3–5) untuk typo/slang
# WORD_NGRAM = (1,2)                # uni+bi (lebih stabil) — ganti ke (2,2) kalau mau bigram-only
# MAX_FEAT_WORD = 30000
# MAX_FEAT_CHAR = 30000
# EMB_MODEL = "distiluse-base-multilingual-cased-v2"
# EMB_NORMALIZE = True
# C_GRID = [0.5, 1.0, 2.0, 3.0]     # grid kecil untuk LinearSVC

# assert TEXT_COL in df.columns and LABEL_COL in df.columns, "Pastikan kolom ada."

# # -------------------
# # Clean ringan
# # -------------------
# def clean_light(s: str) -> str:
#     s = str(s)
#     s = re.sub(r'https?://\S+|www\.\S+', ' ', s)
#     s = re.sub(r'@\w+', ' ', s)
#     s = re.sub(r'[#*_~`]', ' ', s)
#     s = re.sub(r'\s+', ' ', s).strip()
#     return s

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X_all = data[TEXT_COL].astype(str).apply(clean_light).tolist()
# y_txt = data[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# classes = list(le.classes_)
# print("Classes:", classes)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# # -------------------
# # Oversampling MANUAL (train only)
# # -------------------
# def oversample_texts(X, y, random_state=42):
#     rng = np.random.default_rng(random_state)
#     X = np.asarray(X); y = np.asarray(y)
#     cnt = Counter(y); max_n = max(cnt.values())
#     X_out, y_out = [], []
#     for cls, n in cnt.items():
#         idx = np.where(y == cls)[0]
#         add = rng.choice(idx, size=max_n - n, replace=True) if n < max_n else np.array([], dtype=int)
#         take = np.concatenate([idx, add])
#         X_out.extend(X[take].tolist()); y_out.extend([cls]*len(take))
#     perm = rng.permutation(len(y_out))
#     return [X_out[i] for i in perm], np.asarray(y_out)[perm]

# X_train_os, y_train_os = oversample_texts(X_train, y_train, random_state=SEED)
# print("Distribusi train (oversampled):", Counter(y_train_os))

# # -------------------
# # TF-IDF word & char
# # -------------------
# vec_word = TfidfVectorizer(
#     analyzer="word",
#     ngram_range=WORD_NGRAM,
#     min_df=2, max_df=0.95, sublinear_tf=True,
#     max_features=MAX_FEAT_WORD
# )
# Xtr_w = vec_word.fit_transform(X_train_os)
# Xte_w = vec_word.transform(X_test)

# if USE_CHAR:
#     vec_char = TfidfVectorizer(
#         analyzer="char",
#         ngram_range=(3,5),
#         min_df=2, sublinear_tf=True,
#         max_features=MAX_FEAT_CHAR
#     )
#     Xtr_c = vec_char.fit_transform(X_train_os)
#     Xte_c = vec_char.transform(X_test)
# else:
#     # zero-size sparse for easy hstack
#     Xtr_c = csr_matrix((Xtr_w.shape[0], 0))
#     Xte_c = csr_matrix((Xte_w.shape[0], 0))

# # -------------------
# # DistilUSE embeddings
# # -------------------
# st = SentenceTransformer(EMB_MODEL)
# E_tr = st.encode(X_train_os, batch_size=128, show_progress_bar=True,
#                  convert_to_numpy=True, normalize_embeddings=EMB_NORMALIZE)
# E_te = st.encode(X_test, batch_size=128, show_progress_bar=True,
#                  convert_to_numpy=True, normalize_embeddings=EMB_NORMALIZE)

# # pastikan normalized (safety)
# if not EMB_NORMALIZE:
#     E_tr = normalize(E_tr)
#     E_te = normalize(E_te)

# Etr_sp = csr_matrix(E_tr)
# Ete_sp = csr_matrix(E_te)

# # -------------------
# # Stack fitur: [TF-IDF word ; TF-IDF char ; Embeddings]
# # -------------------
# Xtr_stack = hstack([Xtr_w, Xtr_c, Etr_sp], format="csr")
# Xte_stack = hstack([Xte_w, Xte_c, Ete_sp], format="csr")
# print("Shape stack:", Xtr_stack.shape, "->", Xte_stack.shape)

# # -------------------
# # Small CV untuk pilih C terbaik (macro-F1)
# # -------------------
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# best_C, best_cv = None, -1.0

# for C in C_GRID:
#     f1s = []
#     for tr_idx, va_idx in skf.split(Xtr_stack, y_train_os):
#         X_tr, X_va = Xtr_stack[tr_idx], Xtr_stack[va_idx]
#         y_tr, y_va = y_train_os[tr_idx], y_train_os[va_idx]

#         clf = LinearSVC(C=C, class_weight="balanced")
#         clf.fit(X_tr, y_tr)
#         y_hat = clf.predict(X_va)
#         f1s.append(f1_score(y_va, y_hat, average="macro"))
#     mean_cv = float(np.mean(f1s))
#     print(f"C={C} -> CV macro-F1={mean_cv:.4f}")
#     if mean_cv > best_cv:
#         best_cv, best_C = mean_cv, C

# print(f"\nBest C from CV: {best_C} (macro-F1={best_cv:.4f})")

# # -------------------
# # Train final & evaluasi test
# # -------------------
# final_clf = LinearSVC(C=best_C, class_weight="balanced")
# final_clf.fit(Xtr_stack, y_train_os)
# y_pred = final_clf.predict(Xte_stack)

# f1_macro = f1_score(y_test, y_pred, average="macro")
# f1_weighted = f1_score(y_test, y_pred, average="weighted")

# print("\n=== HYBRID (TFIDF word+char + DistilUSE) + LinearSVC ===")
# print("F1_macro   :", round(f1_macro, 6))
# print("F1_weighted:", round(f1_weighted, 6))
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=classes, digits=3))
# print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


In [47]:
# # =========================================================
# # COMPARE SENTENCE EMBEDDINGS + CLASSIFIERS vs FINE-TUNED TRANSFORMER
# # =========================================================
# !pip -q install sentence-transformers xgboost transformers accelerate

# import os, random, numpy as np, pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.utils.class_weight import compute_class_weight
# from sklearn.metrics import accuracy_score, f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
# from xgboost import XGBClassifier
# import torch

# SEED = 42
# random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

# # --------------------------
# # 0) KONFIGURASI KOLUMNAMAS
# # --------------------------
# TEXT_COL  = 'text_final'        # ganti jika perlu: 'text_nostop' atau lainnya
# LABEL_COL = 'emotion'

# assert TEXT_COL in df.columns, f"Kolom teks {TEXT_COL} tidak ada di df."
# assert LABEL_COL in df.columns, f"Kolom label {LABEL_COL} tidak ada di df."

# # bersihkan NaN
# df_ = df[[TEXT_COL, LABEL_COL]].dropna()
# X = df_[TEXT_COL].astype(str).tolist()
# y = df_[LABEL_COL].astype(str).tolist()

# # encode label → id
# le = LabelEncoder()
# y_id = le.fit_transform(y)
# classes = np.unique(y_id)

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y_id, test_size=0.2, random_state=SEED, stratify=y_id
# )

# # helper evaluasi
# def evaluate_preds(y_true, y_pred, name):
#     acc  = accuracy_score(y_true, y_pred)
#     f1m  = f1_score(y_true, y_pred, average='macro')
#     f1w  = f1_score(y_true, y_pred, average='weighted')
#     print(f"\n=== {name} ===")
#     print(classification_report(y_true, y_pred, target_names=le.classes_, digits=3))
#     return {'Model': name, 'Accuracy': acc, 'F1_macro': f1m, 'F1_weighted': f1w}

# results = []

# # =========================================================
# # A) SENTENCE EMBEDDINGS (Sentence-Transformers) + 3 CLASSIFIERS
# # =========================================================
# from sentence_transformers import SentenceTransformer

# # pilih 1–2 encoder multilingual yang ringan & bagus
# ST_MODELS = [
#     "paraphrase-multilingual-MiniLM-L12-v2",
#     "distiluse-base-multilingual-cased-v2",
# ]

# def embed_with(model_name, texts_train, texts_test, normalize=True, bs=128):
#     st = SentenceTransformer(model_name)
#     E_tr = st.encode(texts_train, batch_size=bs, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=normalize)
#     E_te = st.encode(texts_test,  batch_size=bs, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=normalize)
#     return E_tr, E_te

# def train_3_classifiers(E_tr, y_tr, E_te):
#     # class weights untuk imbalance (hanya dipakai di LogisticRegression)
#     cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
#     cw_dict = {int(c): float(w) for c,w in zip(np.unique(y_tr), cw)}

#     # 1) Logistic Regression
#     logreg = LogisticRegression(max_iter=1000, class_weight=cw_dict, n_jobs=-1)
#     logreg.fit(E_tr, y_tr)
#     pred_lr = logreg.predict(E_te)

#     # 2) Linear SVC (margin-based, kuat untuk teks)
#     svc = LinearSVC()
#     svc.fit(E_tr, y_tr)
#     pred_svc = svc.predict(E_te)

#     # 3) XGBoost (kuat menangkap non-linear)
#     xgb = XGBClassifier(
#         n_estimators=400, max_depth=6, learning_rate=0.05,
#         subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
#         tree_method="hist", random_state=SEED, n_jobs=-1
#     )
#     xgb.fit(E_tr, y_tr)
#     pred_xgb = xgb.predict(E_te)

#     return {
#         'LogReg': pred_lr,
#         'LinearSVC': pred_svc,
#         'XGBoost': pred_xgb
#     }

# for st_name in ST_MODELS:
#     print(f"\n>>> Embedding with: {st_name}")
#     E_tr, E_te = embed_with(st_name, X_train, X_test, normalize=True)
#     preds = train_3_classifiers(E_tr, y_train, E_te)
#     for clf_name, y_pred in preds.items():
#         results.append(evaluate_preds(y_test, y_pred, f"{st_name} + {clf_name}"))

# # =========================================================
# # B) FINE-TUNE TRANSFORMER (Opsi B) DENGAN HUGGINGFACE TRAINER
# # =========================================================
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

# FT_MODEL = "indobenchmark/indobert-base-p1"  # bisa ganti 'xlm-roberta-base' jika ingin
# num_labels = len(le.classes_)

# tokenizer = AutoTokenizer.from_pretrained(FT_MODEL)

# class DS(torch.utils.data.Dataset):
#     def __init__(self, texts, labels, tokenizer, max_len=256):
#         self.texts = texts; self.labels = labels; self.tok = tokenizer; self.max_len = max_len
#     def __len__(self): return len(self.texts)
#     def __getitem__(self, i):
#         enc = self.tok(
#             self.texts[i],
#             truncation=True, max_length=self.max_len, padding='max_length', return_tensors='pt'
#         )
#         return {
#             'input_ids': enc['input_ids'].squeeze(0),
#             'attention_mask': enc['attention_mask'].squeeze(0),
#             'labels': torch.tensor(int(self.labels[i]), dtype=torch.long)
#         }

# # split train -> train/val
# X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train)
# ds_tr  = DS(X_tr,  y_tr,  tokenizer)
# ds_val = DS(X_val, y_val, tokenizer)
# ds_te  = DS(X_test, y_test, tokenizer)

# present = np.unique(y_tr)  # hanya kelas yang muncul di train
# cw_present = compute_class_weight(class_weight='balanced', classes=present, y=y_tr)
# cw_map = {int(c): float(w) for c, w in zip(present, cw_present)}

# # bangun vector bobot sepanjang num_labels; default=1.0 untuk kelas yang tidak muncul
# cw_vec = [cw_map.get(i, 1.0) for i in range(num_labels)]
# cw_t = torch.tensor(cw_vec, dtype=torch.float32)

# model = AutoModelForSequenceClassification.from_pretrained(
#     FT_MODEL, num_labels=num_labels, id2label={i:l for i,l in enumerate(le.classes_)}, label2id={l:i for i,l in enumerate(le.classes_)}
# )

# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# class WeightedTrainer(Trainer):
#     def __init__(self, class_weights=None, **kwargs):
#         super().__init__(**kwargs)
#         self.class_weights = class_weights
#     def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs)
#         logits = outputs.logits
#         if self.class_weights is not None:
#             w = self.class_weights.to(logits.device)
#             loss_fn = torch.nn.CrossEntropyLoss(weight=w)
#         else:
#             loss_fn = torch.nn.CrossEntropyLoss()
#         loss = loss_fn(logits, labels)
#         return (loss, outputs) if return_outputs else loss

# args = TrainingArguments(
#     output_dir="./results_ft",
#     num_train_epochs=3,                    # boleh naikkan jika GPU kuat
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     learning_rate=2e-5,
#     warmup_ratio=0.1,
#     weight_decay=0.01,
#     eval_strategy="steps",
#     eval_steps=100,
#     logging_steps=50,
#     save_steps=200,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     greater_is_better=False,
#     report_to="none"
# )

# trainer = WeightedTrainer(
#     class_weights=cw_t,
#     model=model,
#     args=args,
#     train_dataset=ds_tr,
#     eval_dataset=ds_val,
#     data_collator=data_collator,
#     tokenizer=tokenizer
# )

# print("\n>>> Fine-tuning starts…")
# trainer.train()

# # prediksi test
# pred_logits = trainer.predict(ds_te).predictions
# y_pred_ft = pred_logits.argmax(axis=1)
# results.append(evaluate_preds(y_test, y_pred_ft, f"Fine-tuned {FT_MODEL}"))

# # =========================================================
# # RANGKUM HASIL
# # =========================================================
# res_df = pd.DataFrame(results).sort_values("F1_macro", ascending=False).reset_index(drop=True)
# display(res_df)


In [48]:
# # ======================================================
# # distiluse-base-multilingual-cased-v2 + LogisticRegression
# # (pakai text_nostop, opsi oversampling di train)
# # ======================================================
# !pip -q install sentence-transformers

# import os, warnings, random, numpy as np, pandas as pd
# os.environ["TOKENIZERS_PARALLELISM"] = "false"
# warnings.filterwarnings("ignore")

# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from sklearn.linear_model import LogisticRegression
# from collections import Counter
# from sentence_transformers import SentenceTransformer

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# # -------------------
# # KONFIG
# # -------------------
# TEXT_COL  = "text_final"   # kolom teks yang ingin dipakai
# LABEL_COL = "emotion_standardized" if "emotion_standardized" in df.columns else "emotion"
# USE_OVERSAMPLE = True       # set ke False kalau tidak ingin oversampling
# NORMALIZE_EMB  = True       # True = L2-normalized embeddings (umumnya lebih stabil)
# C_VALUE        = 1.0        # regulasi Logistic Regression (bisa coba 0.1 / 3.0 juga)

# assert TEXT_COL in df.columns and LABEL_COL in df.columns, "Kolom teks/label tidak ditemukan."

# # -------------------
# # DATA
# # -------------------
# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X = data[TEXT_COL].astype(str).tolist()
# y_txt = data[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y = le.fit_transform(y_txt)
# print("Classes:", list(le.classes_))

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=SEED, stratify=y
# )

# # -------------------
# # OVERSAMPLING MANUAL (train only, opsional)
# # -------------------
# def oversample_texts(X, y, random_state=42):
#     rng = np.random.default_rng(random_state)
#     X = np.asarray(X); y = np.asarray(y)
#     cnt = Counter(y); max_n = max(cnt.values())
#     X_out, y_out = [], []
#     for cls, n in cnt.items():
#         idx = np.where(y == cls)[0]
#         add = rng.choice(idx, size=max_n - n, replace=True) if n < max_n else np.array([], dtype=int)
#         take = np.concatenate([idx, add])
#         X_out.extend(X[take].tolist()); y_out.extend([cls]*len(take))
#     perm = rng.permutation(len(y_out))
#     return [X_out[i] for i in perm], np.asarray(y_out)[perm]

# if USE_OVERSAMPLE:
#     X_train, y_train = oversample_texts(X_train, y_train, random_state=SEED)
#     print("Distribusi train (oversampled):", Counter(y_train))
# else:
#     print("Distribusi train (original):", Counter(y_train))

# # -------------------
# # EMBEDDINGS (distiluse)
# # -------------------
# MODEL_NAME = "distiluse-base-multilingual-cased-v2"
# st = SentenceTransformer(MODEL_NAME)

# E_train = st.encode(X_train, batch_size=128, show_progress_bar=True,
#                     convert_to_numpy=True, normalize_embeddings=NORMALIZE_EMB)
# E_test  = st.encode(X_test,  batch_size=128, show_progress_bar=True,
#                     convert_to_numpy=True, normalize_embeddings=NORMALIZE_EMB)

# print("Shape embeddings:", E_train.shape, E_test.shape)

# # -------------------
# # LOGISTIC REGRESSION (balanced)
# # -------------------
# clf = LogisticRegression(
#     max_iter=3000,
#     class_weight="balanced",  # bantu kelas minoritas
#     C=C_VALUE,
#     solver="lbfgs",           # stabil untuk L2
#     n_jobs=-1
# )
# clf.fit(E_train, y_train)
# y_pred = clf.predict(E_test)

# # -------------------
# # EVALUASI
# # -------------------
# f1_macro = f1_score(y_test, y_pred, average="macro")
# f1_weighted = f1_score(y_test, y_pred, average="weighted")
# print("\n=== HASIL TEST: distiluse + LogReg ===")
# print("F1_macro   :", round(f1_macro, 6))
# print("F1_weighted:", round(f1_weighted, 6))
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))

# from collections import Counter
# print("Distribusi prediksi:", Counter(y_pred))
# print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))


In [49]:
# # ======================================================
# # distiluse-base-multilingual-cased-v2 + LogisticRegression
# # + (Step 4) Feature-level Improvement:
# #   - Augmentasi minoritas (Back-translation, EDA)
# #   - Hybrid Features: Embeddings + TF-IDF  -> LogReg & XGB
# # ======================================================
# !pip -q install sentence-transformers transformers sentencepiece xgboost

# import os, warnings, random, numpy as np, pandas as pd
# os.environ["TOKENIZERS_PARALLELISM"] = "false"
# warnings.filterwarnings("ignore")

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from sklearn.linear_model import LogisticRegression
# from collections import Counter
# from sentence_transformers import SentenceTransformer

# # TF-IDF & stack
# from sklearn.feature_extraction.text import TfidfVectorizer
# from scipy.sparse import hstack, csr_matrix

# # XGBoost
# from xgboost import XGBClassifier

# # --------------------------------
# # KONFIG
# # --------------------------------
# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# TEXT_COL  = "text_final"
# LABEL_COL = "emotion"

# # 1) Oversampling dasar (manual) — ON/OFF
# USE_OVERSAMPLE = True

# # 2) Sentence embeddings
# EMB_MODEL      = "distiluse-base-multilingual-cased-v2"
# EMB_NORMALIZE  = True          # L2 normalize

# # 3) Logistic Regression (emb-only baseline)
# C_VALUE        = 1.0

# # 4) AUGMENTASI MINORITAS
# USE_BACKTRANS  = False         # True untuk back-translation (butuh runtime)
# BT_PER_CLASS   = 100           # maks sampel/kelas untuk back-translation
# USE_EDA        = True          # augmentasi EDA ringan
# EDA_PER_CLASS  = 100           # maks sampel/kelas untuk EDA

# # 5) HYBRID FEATURES (TF-IDF + Embeddings)
# USE_HYBRID     = True
# TFIDF_WORD_MAX = 30000
# TFIDF_CHAR_MAX = 30000

# assert TEXT_COL in df.columns and LABEL_COL in df.columns, "Kolom teks/label tidak ditemukan."

# # --------------------------------
# # DATA
# # --------------------------------
# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X_all = data[TEXT_COL].astype(str).tolist()
# y_txt = data[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# classes = list(le.classes_)
# print("Classes:", classes)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# # --------------------------------
# # OVERSAMPLING MANUAL (train only, opsional)
# # --------------------------------
# def oversample_texts(X, y, random_state=42):
#     rng = np.random.default_rng(random_state)
#     X = np.asarray(X); y = np.asarray(y)
#     cnt = Counter(y); max_n = max(cnt.values())
#     X_out, y_out = [], []
#     for cls, n in cnt.items():
#         idx = np.where(y == cls)[0]
#         add = rng.choice(idx, size=max_n - n, replace=True) if n < max_n else np.array([], dtype=int)
#         take = np.concatenate([idx, add])
#         X_out.extend(X[take].tolist()); y_out.extend([cls]*len(take))
#     perm = rng.permutation(len(y_out))
#     return [X_out[i] for i in perm], np.asarray(y_out)[perm]

# if USE_OVERSAMPLE:
#     X_train, y_train = oversample_texts(X_train, y_train, random_state=SEED)
#     print("Distribusi train (oversampled):", Counter(y_train))
# else:
#     print("Distribusi train (original):", Counter(y_train))

# # --------------------------------
# # AUGMENTASI MINORITAS
# # --------------------------------
# # (a) Back-translation id<->en dengan MarianMT
# def backtranslate_batch(texts, src_tgt):
#     from transformers import MarianMTModel, MarianTokenizer
#     name = f"Helsinki-NLP/opus-mt-{src_tgt}"
#     tok = MarianTokenizer.from_pretrained(name)
#     mdl = MarianMTModel.from_pretrained(name)
#     out = []
#     bs = 16
#     for i in range(0, len(texts), bs):
#         batch = texts[i:i+bs]
#         enc = tok(batch, return_tensors="pt", truncation=True, padding=True, max_length=256)
#         gen = mdl.generate(**enc, max_length=256)
#         out += tok.batch_decode(gen, skip_special_tokens=True)
#     return out

# def backtranslate_id_en_id(texts):
#     en = backtranslate_batch(texts, "id-en")
#     bt = backtranslate_batch(en, "en-id")
#     return bt

# # (b) EDA ringan: random swap/insert/delete + synonym map mini
# import re
# def tokenize_simple(s): return [t for t in re.findall(r'\w+', s.lower()) if len(t) > 2]
# SYN_MAP = {
#     "bagus":["keren","mantap","hebat"], "jelek":["buruk","gaenak","parah"],
#     "marah":["kesal","geram"], "senang":["gembira","bahagia","suka"],
#     "takut":["ngeri","khawatir"], "sedih":["murung","kecewa"],
# }
# def synonym_replace(tokens, p=0.2):
#     out=[]
#     for t in tokens:
#         if t in SYN_MAP and random.random()<p:
#             out.append(random.choice(SYN_MAP[t]))
#         else:
#             out.append(t)
#     return out
# def random_swap(tokens, n=1):
#     toks=tokens[:]
#     for _ in range(n):
#         if len(toks)<2: break
#         i,j = random.sample(range(len(toks)),2)
#         toks[i],toks[j]=toks[j],toks[i]
#     return toks
# def random_insert(tokens, n=1):
#     toks=tokens[:]
#     for _ in range(n):
#         if not toks: break
#         pos = random.randrange(len(toks))
#         toks.insert(pos, random.choice(toks))
#     return toks
# def random_delete(tokens, p=0.1):
#     out=[t for t in tokens if random.random()>p]
#     return out if out else tokens
# def eda_augment(text, ops=("syn","swap","insert","delete")):
#     toks = tokenize_simple(text)
#     if not toks: return text
#     if "syn" in ops:    toks = synonym_replace(toks, p=0.2)
#     if "swap" in ops:   toks = random_swap(toks, n=1)
#     if "insert" in ops: toks = random_insert(toks, n=1)
#     if "delete" in ops: toks = random_delete(toks, p=0.1)
#     return " ".join(toks)

# def augment_minority(X, y, use_bt=False, bt_per_class=50, use_eda=True, eda_per_class=100):
#     X_aug, y_aug = [], []
#     y_arr = np.asarray(y); X_arr = np.asarray(X)
#     counts = Counter(y_arr)
#     median_n = int(np.median(list(counts.values())))
#     minor = [c for c,n in counts.items() if n < median_n]
#     print("Minority classes:", {int(c): int(counts[c]) for c in minor})

#     for cls in minor:
#         idx = np.where(y_arr==cls)[0].tolist()
#         random.shuffle(idx)
#         # Back-translation
#         if use_bt and idx:
#             take = idx[:min(bt_per_class, len(idx))]
#             bt_texts = backtranslate_id_en_id(X_arr[take].tolist())
#             X_aug.extend(bt_texts); y_aug.extend([cls]*len(bt_texts))
#         # EDA
#         if use_eda and idx:
#             take = idx[:min(eda_per_class, len(idx))]
#             eda_texts = [eda_augment(X_arr[i]) for i in take]
#             X_aug.extend(eda_texts); y_aug.extend([cls]*len(eda_texts))
#     return X_aug, y_aug

# # Jalankan augmentasi (opsional)
# X_aug, y_aug = [], []
# if USE_BACKTRANS or USE_EDA:
#     X_aug, y_aug = augment_minority(
#         X_train, y_train,
#         use_bt=USE_BACKTRANS, bt_per_class=BT_PER_CLASS,
#         use_eda=USE_EDA, eda_per_class=EDA_PER_CLASS
#     )
#     if X_aug:
#         X_train = X_train + X_aug
#         y_train = list(y_train) + y_aug
#         print("Total train setelah augment:", len(y_train), Counter(y_train))

# # --------------------------------
# # EMBEDDINGS (distiluse) untuk baseline & hybrid
# # --------------------------------
# st = SentenceTransformer(EMB_MODEL)
# E_train = st.encode(X_train, batch_size=128, show_progress_bar=True,
#                     convert_to_numpy=True, normalize_embeddings=EMB_NORMALIZE)
# E_test  = st.encode(X_test,  batch_size=128, show_progress_bar=True,
#                     convert_to_numpy=True, normalize_embeddings=EMB_NORMALIZE)
# print("Shape embeddings:", E_train.shape, E_test.shape)

# # --------------------------------
# # BASELINE: Embeddings + LogReg
# # --------------------------------
# clf_emb = LogisticRegression(
#     max_iter=3000, class_weight="balanced",
#     C=C_VALUE, solver="lbfgs", n_jobs=-1
# )
# clf_emb.fit(E_train, y_train)
# y_pred_emb = clf_emb.predict(E_test)

# print("\n=== HASIL TEST: distiluse + LogReg (emb-only) ===")
# print("F1_macro   :", round(f1_score(y_test, y_pred_emb, average="macro"), 6))
# print("F1_weighted:", round(f1_score(y_test, y_pred_emb, average="weighted"), 6))
# print("\nClassification Report (emb-only):")
# print(classification_report(y_test, y_pred_emb, target_names=classes, digits=3))

# # --------------------------------
# # HYBRID: TF-IDF(word+char) + Embeddings  -> LogReg & XGB
# # --------------------------------
# if USE_HYBRID:
#     tfidf_word = TfidfVectorizer(
#         analyzer="word", ngram_range=(1,2),
#         min_df=2, max_df=0.95, sublinear_tf=True,
#         max_features=TFIDF_WORD_MAX
#     )
#     tfidf_char = TfidfVectorizer(
#         analyzer="char", ngram_range=(3,5),
#         min_df=2, sublinear_tf=True,
#         max_features=TFIDF_CHAR_MAX
#     )

#     Xtr_w = tfidf_word.fit_transform(X_train)
#     Xte_w = tfidf_word.transform(X_test)
#     Xtr_c = tfidf_char.fit_transform(X_train)
#     Xte_c = tfidf_char.transform(X_test)

#     # stack: [TFIDF_word ; TFIDF_char ; Embeddings]
#     E_tr_sp = csr_matrix(E_train)
#     E_te_sp = csr_matrix(E_test)
#     Xtr_stack = hstack([Xtr_w, Xtr_c, E_tr_sp], format="csr")
#     Xte_stack = hstack([Xte_w, Xte_c, E_te_sp], format="csr")
#     print("Shapes -> word:", Xtr_w.shape, "char:", Xtr_c.shape, "emb:", E_tr_sp.shape, "stack:", Xtr_stack.shape)

#     # (i) LogReg di hybrid
#     clf_h_logreg = LogisticRegression(
#         max_iter=3000, class_weight="balanced",
#         C=1.0, solver="lbfgs", n_jobs=-1
#     )
#     clf_h_logreg.fit(Xtr_stack, y_train)
#     y_pred_h_lr = clf_h_logreg.predict(Xte_stack)

#     print("\n=== HYBRID (TFIDF + Embeddings) + LogReg ===")
#     print("F1_macro   :", round(f1_score(y_test, y_pred_h_lr, average="macro"), 6))
#     print("F1_weighted:", round(f1_score(y_test, y_pred_h_lr, average="weighted"), 6))
#     print("\nClassification Report (hybrid-LogReg):")
#     print(classification_report(y_test, y_pred_h_lr, target_names=classes, digits=3))

#     # (ii) XGBoost di hybrid
#     clf_h_xgb = XGBClassifier(
#         n_estimators=400, learning_rate=0.08,
#         max_depth=8, subsample=0.9, colsample_bytree=0.9,
#         eval_metric="mlogloss", random_state=SEED, n_jobs=-1
#     )
#     clf_h_xgb.fit(Xtr_stack, y_train)
#     y_pred_h_xgb = clf_h_xgb.predict(Xte_stack)

#     print("\n=== HYBRID (TFIDF + Embeddings) + XGBoost ===")
#     print("F1_macro   :", round(f1_score(y_test, y_pred_h_xgb, average="macro"), 6))
#     print("F1_weighted:", round(f1_score(y_test, y_pred_h_xgb, average="weighted"), 6))
#     print("\nClassification Report (hybrid-XGB):")
#     print(classification_report(y_test, y_pred_h_xgb, target_names=classes, digits=3))

# # --------------------------------
# # Rangkuman singkat
# # --------------------------------
# from pandas import DataFrame
# rows = [
#     ("Emb-only + LogReg", f1_score(y_test, y_pred_emb, average="macro"), f1_score(y_test, y_pred_emb, average="weighted")),
# ]
# if USE_HYBRID:
#     rows += [
#         ("Hybrid TFIDF+Emb + LogReg", f1_score(y_test, y_pred_h_lr, average="macro"), f1_score(y_test, y_pred_h_lr, average="weighted")),
#         ("Hybrid TFIDF+Emb + XGB",    f1_score(y_test, y_pred_h_xgb, average="macro"), f1_score(y_test, y_pred_h_xgb, average="weighted")),
#     ]
# print("\n=== Rangkuman ===")
# print(DataFrame(rows, columns=["Model","F1_macro","F1_weighted"]))
# print("\nConfusion matrix (emb-only):\n", confusion_matrix(y_test, y_pred_emb))


In [50]:
# # ================================================
# # HYBRID: TF-IDF (word+char) + distiluse embeddings -> Logistic Regression
# # ================================================
# !pip -q install sentence-transformers
 
# import os, warnings, random, numpy as np, pandas as pd
# os.environ["TOKENIZERS_PARALLELISM"] = "false"
# warnings.filterwarnings("ignore")

# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from sklearn.linear_model import LogisticRegression
# from sklearn.feature_extraction.text import TfidfVectorizer
# from scipy.sparse import hstack, csr_matrix
# from sentence_transformers import SentenceTransformer
# from collections import Counter

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# # -------------------
# # KONFIG
# # -------------------
# TEXT_COL  = "text_final"
# LABEL_COL = "emotion_standardized" if "emotion_standardized" in df.columns else "emotion"
# USE_OVERSAMPLE = True
# EMB_NORMALIZE  = True
# C_GRID         = [0.3, 1.0, 3.0]  # grid kecil untuk LR

# assert TEXT_COL in df.columns and LABEL_COL in df.columns, "Kolom teks/label tidak ditemukan."

# data = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X = data[TEXT_COL].astype(str).tolist()
# y_txt = data[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y = le.fit_transform(y_txt)
# print("Classes:", list(le.classes_))

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, random_state=SEED, stratify=y
# )

# # # -------------------
# # # Oversampling MANUAL (train only)
# # # -------------------
# # def oversample_texts(X, y, random_state=42):
# #     rng = np.random.default_rng(random_state)
# #     X = np.asarray(X); y = np.asarray(y)
# #     cnt = Counter(y); max_n = max(cnt.values())
# #     X_out, y_out = [], []
# #     for cls, n in cnt.items():
# #         idx = np.where(y == cls)[0]
# #         add = rng.choice(idx, size=max_n - n, replace=True) if n < max_n else np.array([], dtype=int)
# #         take = np.concatenate([idx, add])
# #         X_out.extend(X[take].tolist()); y_out.extend([cls]*len(take))
# #     perm = rng.permutation(len(y_out))
# #     return [X_out[i] for i in perm], np.asarray(y_out)[perm]

# # if USE_OVERSAMPLE:
# #     X_train_os, y_train_os = oversample_texts(X_train, y_train, random_state=SEED)
# #     print("Distribusi train (oversampled):", Counter(y_train_os))
# # else:
# #     X_train_os, y_train_os = X_train, y_train
# #     print("Distribusi train (original):", Counter(y_train_os))

# # -------------------
# # TF-IDF (word & char)
# # -------------------
# tfidf_word = TfidfVectorizer(
#     analyzer="word", ngram_range=(1,2),
#     min_df=2, max_df=0.95, sublinear_tf=True,
#     max_features=20000  # batasin biar tidak terlalu besar
# )
# Xtr_w = tfidf_word.fit_transform(X_train_os)
# Xte_w = tfidf_word.transform(X_test)
# Xtr_c = tfidf_char.fit_transform(X_train_os)
# Xte_c = tfidf_char.transform(X_test)

# # -------------------
# # distiluse embeddings
# # -------------------
# MODEL_NAME = "distiluse-base-multilingual-cased-v2"
# st = SentenceTransformer(MODEL_NAME)

# E_tr = st.encode(X_train_os, batch_size=128, show_progress_bar=True,
#                  convert_to_numpy=True, normalize_embeddings=EMB_NORMALIZE)
# E_te = st.encode(X_test, batch_size=128, show_progress_bar=True,
#                  convert_to_numpy=True, normalize_embeddings=EMB_NORMALIZE)

# # ubah dense -> sparse biar bisa hstack
# E_tr_sp = csr_matrix(E_tr)
# E_te_sp = csr_matrix(E_te)

# # -------------------
# # stack fitur: [TFIDF_word ; TFIDF_char ; Embeddings]
# # -------------------
# Xtr_stack = hstack([Xtr_w, Xtr_c, E_tr_sp], format="csr")
# Xte_stack = hstack([Xte_w, Xte_c, E_te_sp], format="csr")

# print("Shapes -> word:", Xtr_w.shape, "char:", Xtr_c.shape, "emb:", E_tr_sp.shape, "stack:", Xtr_stack.shape)

# # -------------------
# # CV kecil untuk pilih C terbaik (macro-F1)
# # -------------------
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# best_C, best_cv = None, -1

# for C in C_GRID:
#     f1s = []
#     for tr_idx, va_idx in skf.split(Xtr_stack, y_train_os):
#         X_tr, X_va = Xtr_stack[tr_idx], Xtr_stack[va_idx]
#         y_tr, y_va = y_train_os[tr_idx], y_train_os[va_idx]

#         clf = LogisticRegression(
#             max_iter=3000,
#             class_weight="balanced",
#             C=C,
#             solver="lbfgs",  # L2; stabil untuk fitur besar
#             n_jobs=-1
#         )
#         clf.fit(X_tr, y_tr)
#         y_hat = clf.predict(X_va)
#         f1s.append(f1_score(y_va, y_hat, average="macro"))
#     mean_f1 = float(np.mean(f1s))
#     print(f"C={C} -> CV macro-F1={mean_f1:.4f}")
#     if mean_f1 > best_cv:
#         best_cv, best_C = mean_f1, C

# print(f"\nBest C from CV: {best_C} (macro-F1={best_cv:.4f})")

# # -------------------
# # Train final di train_os & evaluasi di test
# # -------------------
# final_clf = LogisticRegression(
#     max_iter=3000,
#     class_weight="balanced",
#     C=best_C,
#     solver="lbfgs",
#     n_jobs=-1
# )
# final_clf.fit(Xtr_stack, y_train_os)
# y_pred = final_clf.predict(Xte_stack)

# f1_macro = f1_score(y_test, y_pred, average="macro")
# f1_weighted = f1_score(y_test, y_pred, average="weighted")

# print("\n=== HYBRID (TFIDF word+char + distiluse) + LogReg ===")
# print("F1_macro   :", round(f1_macro, 6))
# print("F1_weighted:", round(f1_weighted, 6))
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred, target_names=le.classes_, digits=3))
# print("Distribusi prediksi:", Counter(y_pred))
# print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))


In [51]:
# # ============================
# # Sentence Embeddings (distiluse) + Logistic Regression + Optuna (50 trials) + optional SVD
# # ============================
# !pip -q install optuna sentence-transformers

# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sklearn.decomposition import TruncatedSVD
# from sklearn.utils.class_weight import compute_class_weight
# from sentence_transformers import SentenceTransformer
# import optuna
# from optuna.pruners import MedianPruner
# import warnings, os, random
# warnings.filterwarnings("ignore")

# # --------- KONFIG DATA ---------
# SEED = 42
# random.seed(SEED); np.random.seed(SEED)
# TEXT_COL  = "text_final"  # ganti sesuai kolom teks bersihmu
# LABEL_COL = "emotion_standardized" if "emotion_standardized" in df.columns else "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns

# df_ = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X_all = df_[TEXT_COL].astype(str).tolist()
# y_txt = df_[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# num_labels = len(le.classes_)
# print("Classes:", list(le.classes_))

# # stratified split
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# # # --------- EMBEDDINGS (encode sekali agar cepat) ---------
# # MODEL_NAME = "distiluse-base-multilingual-cased-v2"
# # st = SentenceTransformer(MODEL_NAME)

# # E_train_norm = st.encode(X_train, batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
# # E_test_norm  = st.encode(X_test,  batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

# # E_train_raw  = st.encode(X_train, batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)
# # E_test_raw   = st.encode(X_test,  batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)

# # print("Embeddings:", E_train_norm.shape, E_train_raw.shape)

# # # --------- helper SVD opsional ---------
# # def maybe_svd_fit_transform(X, trial):
# #     use_svd = trial.suggest_categorical("use_svd", [True, False])
# #     if not use_svd:
# #         return X, None, False
# #     n_feat = X.shape[1]
# #     n_comp = trial.suggest_int("svd_n_components", 64, min(512, max(64, n_feat)), step=32)
# #     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
# #     Xr = svd.fit_transform(X)
# #     postnorm = trial.suggest_categorical("svd_post_l2norm", [True, False])
# #     if postnorm:
# #         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
# #     return Xr, svd, postnorm

# # def maybe_svd_transform(X, svd, postnorm):
# #     if svd is None:
# #         return X
# #     Xr = svd.transform(X)
# #     if postnorm:
# #         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
# #     return Xr

# # # --------- Objective Optuna: 5-fold CV macro-F1 ---------
# # def objective(trial: optuna.Trial) -> float:
# #     # pilih normalized/non-normalized embeddings
# #     use_norm = trial.suggest_categorical("use_normalized_embeddings", [True, False])
# #     Xmat = E_train_norm if use_norm else E_train_raw

# #     # opsi SVD
# #     Xmat, svd, postnorm = maybe_svd_fit_transform(Xmat, trial)

# #     # hyperparams LR
# #     solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"])
# #     class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
# #     C = trial.suggest_float("C", 1e-4, 1e+4, log=True)

# #     # penalty valid per solver
# #     if solver == "lbfgs":
# #         penalty, l1_ratio = "l2", None
# #     elif solver == "liblinear":
# #         penalty = trial.suggest_categorical("penalty_liblinear", ["l1", "l2"])
# #         l1_ratio = None
# #     else:  # saga
# #         penalty = trial.suggest_categorical("penalty_saga", ["l1", "l2", "elasticnet"])
# #         l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None
# #         if penalty == "elasticnet" and solver != "saga":
# #             raise optuna.TrialPruned()

# #     # 5-fold CV
# #     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
# #     f1s = []

# #     for tr_idx, va_idx in skf.split(Xmat, y_train):
# #         X_tr, X_va = Xmat[tr_idx], Xmat[va_idx]
# #         y_tr, y_va = np.array(y_train)[tr_idx], np.array(y_train)[va_idx]

# #         # inisialisasi LR
# #         clf = LogisticRegression(
# #             solver=solver,
# #             penalty=penalty,
# #             C=C,
# #             class_weight=class_weight,
# #             l1_ratio=l1_ratio,
# #             max_iter=3000,
# #             n_jobs=-1 if solver != "liblinear" else None,
# #             random_state=SEED
# #         )

# #         clf.fit(X_tr, y_tr)
# #         y_pred = clf.predict(X_va)
# #         f1s.append(f1_score(y_va, y_pred, average="macro"))

# #     return float(np.mean(f1s))

# # study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# # study.optimize(objective, n_trials=50, show_progress_bar=True)

# # print("\nBest trial #", study.best_trial.number)
# # print("Best CV macro-F1:", round(study.best_value, 6))
# # print("Best params:", study.best_params)

# # # --------- Train final dengan best params + evaluasi test ---------
# # best = study.best_params
# # use_norm = best.get("use_normalized_embeddings", True)
# # X_tr_all = E_train_norm if use_norm else E_train_raw
# # X_te_all = E_test_norm  if use_norm else E_test_raw

# # # apply SVD jika dipilih
# # if best.get("use_svd", False):
# #     n_comp = best["svd_n_components"]
# #     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
# #     X_tr_final = svd.fit_transform(X_tr_all)
# #     X_te_final = svd.transform(X_te_all)
# #     if best.get("svd_post_l2norm", False):
# #         X_tr_final = X_tr_final / (np.linalg.norm(X_tr_final, axis=1, keepdims=True) + 1e-12)
# #         X_te_final = X_te_final / (np.linalg.norm(X_te_final, axis=1, keepdims=True) + 1e-12)
# # else:
# #     X_tr_final, X_te_final = X_tr_all, X_te_all

# # # reconstruct LR hyperparams
# # solver = best["solver"]
# # class_weight = best["class_weight"]
# # C = best["C"]
# # if solver == "lbfgs":
# #     penalty, l1_ratio = "l2", None
# # elif solver == "liblinear":
# #     penalty, l1_ratio = best.get("penalty_liblinear", "l2"), None
# # else:
# #     penalty, l1_ratio = best.get("penalty_saga", "l2"), best.get("l1_ratio", None)

# # final_clf = LogisticRegression(
# #     solver=solver,
# #     penalty=penalty,
# #     C=C,
# #     class_weight=class_weight,
# #     l1_ratio=l1_ratio,
# #     max_iter=3000,
# #     n_jobs=-1 if solver != "liblinear" else None,
# #     random_state=SEED
# # )
# # final_clf.fit(X_tr_final, y_train)
# # y_pred_test = final_clf.predict(X_te_final)

# # f1_macro = f1_score(y_test, y_pred_test, average="macro")
# # f1_weighted = f1_score(y_test, y_pred_test, average="weighted")
# # print("\n=== Test set (Optuna-best) ===")
# # print("F1_macro   :", round(f1_macro, 6))
# # print("F1_weighted:", round(f1_weighted, 6))
# # print(classification_report(y_test, y_pred_test, target_names=le.classes_, digits=3))


In [52]:
# # =========================================================
# # C) TF-IDF + 4 CLASSIFIERS (LogReg, LinearSVC, ComplementNB, XGBoost)
# # =========================================================
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
# from sklearn.naive_bayes import ComplementNB
# from sklearn.utils.class_weight import compute_class_weight
# from xgboost import XGBClassifier
# import numpy as np

# # Dua varian TF-IDF: word & char n-grams
# TFIDF_SETUPS = [
#     ("TFIDF(word 1-2)",  TfidfVectorizer(
#         analyzer="word", ngram_range=(1,2),
#         min_df=2, sublinear_tf=True, max_features=None
#     )),
#     ("TFIDF(char 3-5)",  TfidfVectorizer(
#         analyzer="char", ngram_range=(3,5),
#         min_df=2, sublinear_tf=True, max_features=None
#     )),
# ]

# # sample_weight untuk XGBoost (imbalance-aware)
# classes_present = np.unique(y_train)
# cw_vals = compute_class_weight(class_weight='balanced', classes=classes_present, y=y_train)
# cw_map = {int(c): float(w) for c, w in zip(classes_present, cw_vals)}
# sample_weight_xgb = np.array([cw_map.get(int(y), 1.0) for y in y_train], dtype=float)

# def run_tfidf_block(name, vectorizer, X_tr_text, y_tr, X_te_text):
#     # Fit TF-IDF pada train, transform train & test
#     Xtr = vectorizer.fit_transform(X_tr_text)
#     Xte = vectorizer.transform(X_te_text)

#     # 1) Logistic Regression
#     logreg = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight='balanced')
#     logreg.fit(Xtr, y_tr)
#     y_pred_lr = logreg.predict(Xte)
#     results.append(evaluate_preds(y_test, y_pred_lr, f"{name} + LogReg"))

#     # 2) Linear SVC
#     svc = LinearSVC()  # bisa gunakan class_weight='balanced' jika diperlukan
#     svc.fit(Xtr, y_tr)
#     y_pred_svc = svc.predict(Xte)
#     results.append(evaluate_preds(y_test, y_pred_svc, f"{name} + LinearSVC"))

#     # 3) Complement Naive Bayes
#     cnb = ComplementNB()
#     cnb.fit(Xtr, y_tr)
#     y_pred_cnb = cnb.predict(Xte)
#     results.append(evaluate_preds(y_test, y_pred_cnb, f"{name} + ComplementNB"))

#     # 4) XGBoost (multi-class, softprob → argmax). Bekerja langsung dengan matriks sparse TF-IDF.
#     n_classes = len(np.unique(y_train))
#     xgb = XGBClassifier(
#         objective='multi:softprob',
#         num_class=n_classes,
#         n_estimators=600,
#         learning_rate=0.05,
#         max_depth=6,
#         subsample=0.9,
#         colsample_bytree=0.9,
#         reg_lambda=1.0,
#         tree_method="hist",   # ganti "gpu_hist" jika pakai GPU
#         random_state=42,
#         n_jobs=-1
#     )
#     xgb.fit(Xtr, y_tr,
#             sample_weight=sample_weight_xgb,
#             eval_set=[(Xtr, y_tr), (Xte, y_test)],
#             eval_metric='mlogloss',
#             verbose=False)
#     y_pred_xgb = xgb.predict(Xte)
#     results.append(evaluate_preds(y_test, y_pred_xgb, f"{name} + XGBoost"))

# for tf_name, tf_vec in TFIDF_SETUPS:
#     print(f"\n>>> TF-IDF variant: {tf_name}")
#     run_tfidf_block(tf_name, tf_vec, X_train, y_train, X_test)

# # Tampilkan rangkuman semua hasil (A+B+C)
# res_df = pd.DataFrame(results).sort_values("F1_macro", ascending=False).reset_index(drop=True)
# display(res_df)


In [53]:
# # ============================
# # Sentence Embeddings (distiluse) + Logistic Regression + Optuna (50 trials) + optional SVD
# # ============================
# !pip -q install optuna sentence-transformers

# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sklearn.decomposition import TruncatedSVD
# from sklearn.utils.class_weight import compute_class_weight
# from sentence_transformers import SentenceTransformer
# import optuna
# from optuna.pruners import MedianPruner
# import warnings, os, random
# warnings.filterwarnings("ignore")

# # --------- KONFIG DATA ---------
# SEED = 42
# random.seed(SEED); np.random.seed(SEED)
# TEXT_COL  = "text_nostop"  # ganti sesuai kolom teks bersihmu
# LABEL_COL = "emotion_standardized" if "emotion_standardized" in df.columns else "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns

# df_ = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# X_all = df_[TEXT_COL].astype(str).tolist()
# y_txt = df_[LABEL_COL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# num_labels = len(le.classes_)
# print("Classes:", list(le.classes_))

# # stratified split
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# # --------- EMBEDDINGS (encode sekali agar cepat) ---------
# MODEL_NAME = "distiluse-base-multilingual-cased-v2"
# st = SentenceTransformer(MODEL_NAME)

# E_train_norm = st.encode(X_train, batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
# E_test_norm  = st.encode(X_test,  batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

# E_train_raw  = st.encode(X_train, batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)
# E_test_raw   = st.encode(X_test,  batch_size=128, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=False)

# print("Embeddings:", E_train_norm.shape, E_train_raw.shape)

# # --------- helper SVD opsional ---------
# def maybe_svd_fit_transform(X, trial):
#     use_svd = trial.suggest_categorical("use_svd", [True, False])
#     if not use_svd:
#         return X, None, False
#     n_feat = X.shape[1]
#     n_comp = trial.suggest_int("svd_n_components", 64, min(512, max(64, n_feat)), step=32)
#     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
#     Xr = svd.fit_transform(X)
#     postnorm = trial.suggest_categorical("svd_post_l2norm", [True, False])
#     if postnorm:
#         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
#     return Xr, svd, postnorm

# def maybe_svd_transform(X, svd, postnorm):
#     if svd is None:
#         return X
#     Xr = svd.transform(X)
#     if postnorm:
#         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
#     return Xr

# # --------- Objective Optuna: 5-fold CV macro-F1 ---------
# def objective(trial: optuna.Trial) -> float:
#     # pilih normalized/non-normalized embeddings
#     use_norm = trial.suggest_categorical("use_normalized_embeddings", [True, False])
#     Xmat = E_train_norm if use_norm else E_train_raw

#     # opsi SVD
#     Xmat, svd, postnorm = maybe_svd_fit_transform(Xmat, trial)

#     # hyperparams LR
#     solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"])
#     class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
#     C = trial.suggest_float("C", 1e-5, 1e+5, log=True)

#     # penalty valid per solver
#     if solver == "lbfgs":
#         penalty, l1_ratio = "l2", None
#     elif solver == "liblinear":
#         penalty = trial.suggest_categorical("penalty_liblinear", ["l1", "l2"])
#         l1_ratio = None
#     else:  # saga
#         penalty = trial.suggest_categorical("penalty_saga", ["l1", "l2", "elasticnet"])
#         l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None
#         if penalty == "elasticnet" and solver != "saga":
#             raise optuna.TrialPruned()

#     # 5-fold CV
#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
#     f1s = []

#     for tr_idx, va_idx in skf.split(Xmat, y_train):
#         X_tr, X_va = Xmat[tr_idx], Xmat[va_idx]
#         y_tr, y_va = np.array(y_train)[tr_idx], np.array(y_train)[va_idx]

#         # inisialisasi LR
#         clf = LogisticRegression(
#             solver=solver,
#             penalty=penalty,
#             C=C,
#             class_weight=class_weight,
#             l1_ratio=l1_ratio,
#             max_iter=3000,
#             n_jobs=-1 if solver != "liblinear" else None,
#             random_state=SEED
#         )

#         clf.fit(X_tr, y_tr)
#         y_pred = clf.predict(X_va)
#         f1s.append(f1_score(y_va, y_pred, average="macro"))

#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=100, show_progress_bar=True)

# print("\nBest trial #", study.best_trial.number)
# print("Best CV macro-F1:", round(study.best_value, 6))
# print("Best params:", study.best_params)

# # --------- Train final dengan best params + evaluasi test ---------
# best = study.best_params
# use_norm = best.get("use_normalized_embeddings", True)
# X_tr_all = E_train_norm if use_norm else E_train_raw
# X_te_all = E_test_norm  if use_norm else E_test_raw

# # apply SVD jika dipilih
# if best.get("use_svd", False):
#     n_comp = best["svd_n_components"]
#     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
#     X_tr_final = svd.fit_transform(X_tr_all)
#     X_te_final = svd.transform(X_te_all)
#     if best.get("svd_post_l2norm", False):
#         X_tr_final = X_tr_final / (np.linalg.norm(X_tr_final, axis=1, keepdims=True) + 1e-12)
#         X_te_final = X_te_final / (np.linalg.norm(X_te_final, axis=1, keepdims=True) + 1e-12)
# else:
#     X_tr_final, X_te_final = X_tr_all, X_te_all

# # reconstruct LR hyperparams
# solver = best["solver"]
# class_weight = best["class_weight"]
# C = best["C"]
# if solver == "lbfgs":
#     penalty, l1_ratio = "l2", None
# elif solver == "liblinear":
#     penalty, l1_ratio = best.get("penalty_liblinear", "l2"), None
# else:
#     penalty, l1_ratio = best.get("penalty_saga", "l2"), best.get("l1_ratio", None)

# final_clf = LogisticRegression(
#     solver=solver,
#     penalty=penalty,
#     C=C,
#     class_weight=class_weight,
#     l1_ratio=l1_ratio,
#     max_iter=3000,
#     n_jobs=-1 if solver != "liblinear" else None,
#     random_state=SEED
# )
# final_clf.fit(X_tr_final, y_train)
# y_pred_test = final_clf.predict(X_te_final)

# f1_macro = f1_score(y_test, y_pred_test, average="macro")
# f1_weighted = f1_score(y_test, y_pred_test, average="weighted")
# print("\n=== Test set (Optuna-best) ===")
# print("F1_macro   :", round(f1_macro, 6))
# print("F1_weighted:", round(f1_weighted, 6))
# print(classification_report(y_test, y_pred_test, target_names=le.classes_, digits=3))


In [54]:
# # =========================
# # SetFit fine-tuning (sentence-transformers) on text_nostop
# # =========================
# !pip -q install setfit

# import numpy as np, pandas as pd, random
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report, confusion_matrix
# from setfit import SetFitModel, SetFitTrainer, sample_dataset
# from datasets import Dataset

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# TEXT_COL  = "text_nostop"
# LABEL_COL = "emotion_standardized" if "emotion_standardized" in df.columns else "emotion"
# assert TEXT_COL in df.columns and LABEL_COL in df.columns

# # data -> HuggingFace Dataset
# df_local = df[[TEXT_COL, LABEL_COL]].dropna().copy()
# le = LabelEncoder()
# df_local["label_id"] = le.fit_transform(df_local[LABEL_COL].astype(str))
# train_df, test_df = train_test_split(df_local, test_size=0.2, random_state=SEED, stratify=df_local["label_id"])

# ds_train = Dataset.from_pandas(train_df[[TEXT_COL, "label_id"]].rename(columns={TEXT_COL:"text"}), preserve_index=False)
# ds_test  = Dataset.from_pandas(test_df [[TEXT_COL, "label_id"]].rename(columns={TEXT_COL:"text"}), preserve_index=False)

# # (opsional) balancing kecil via SetFit sampler (ambil N per kelas)
# # ds_train_bal = sample_dataset(ds_train, label_column="label_id", num_samples=200, seed=SEED)
# ds_train_bal = ds_train

# # Backbone: coba yang lebih kuat dulu
# BACKBONE = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # atau "distiluse-base-multilingual-cased-v2"

# model = SetFitModel.from_pretrained(BACKBONE)

# trainer = SetFitTrainer(
#     model=model,
#     train_dataset=ds_train_bal,
#     eval_dataset=ds_test,
#     loss_class=None,                     # default cosine contastive + LogReg head
#     metric="f1",
#     batch_size=32,
#     num_iterations=20,                   # steps contrastive per epoch
#     num_epochs=5,                        # naikkan ke 8–10 jika masih underfit
#     learning_rate=2e-5,
#     seed=SEED,
#     column_mapping={"text":"text","label_id":"label"},
#     use_text_augmentation=True,          # augmentasi pasangan (hard negatives)
#     sampling_strategy="balanced"         # penting untuk imbalance
# )

# trainer.train()
# metrics = trainer.evaluate()
# print("SetFit (eval on val/test ds_test) F1:", metrics)

# # Prediksi di test set untuk laporan lengkap
# preds = trainer.model.predict(test_df[TEXT_COL].tolist())
# print("\n=== SETFIT REPORT (mpnet) ===")
# print("F1_macro   :", f1_score(test_df["label_id"], preds, average="macro"))
# print("F1_weighted:", f1_score(test_df["label_id"], preds, average="weighted"))
# print(classification_report(test_df["label_id"], preds, target_names=le.classes_, digits=3))
# print("Confusion:\n", confusion_matrix(test_df["label_id"], preds))


In [55]:
# # ============================================
# # distiluse + Logistic Regression + Optuna (50 trials) + SVD (opsional)
# # + Gabung text_final & text_nostop + Oversampling MANUAL (tanpa imblearn)
# # ============================================
# !pip -q install optuna sentence-transformers

# import os, warnings, random
# os.environ["TOKENIZERS_PARALLELISM"] = "false"
# warnings.filterwarnings("ignore")

# import numpy as np
# import pandas as pd
# from collections import Counter
# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sklearn.decomposition import TruncatedSVD
# from sentence_transformers import SentenceTransformer
# import optuna
# from optuna.pruners import MedianPruner

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# # ---------- 1) Siapkan data ----------
# TEXT_A = "text_final"   # teks bersih (normalisasi/spell)
# TEXT_B = "text_nostop"  # versi tanpa stopwords
# LABEL  = "emotion_standardized" if "emotion_standardized" in df.columns else "emotion"
# assert TEXT_A in df.columns and TEXT_B in df.columns and LABEL in df.columns, "Kolom tidak lengkap."

# def combine_text(a, b):
#     a = str(a) if pd.notna(a) else ""
#     b = str(b) if pd.notna(b) else ""
#     return f"{a} || {b}" if (a or b) else ""

# df_ = df[[TEXT_A, TEXT_B, LABEL]].dropna().copy()
# df_["text_combo"] = [combine_text(a, b) for a, b in zip(df_[TEXT_A], df_[TEXT_B])]

# X_all = df_["text_combo"].astype(str).tolist()
# y_txt = df_[LABEL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# print("Classes:", list(le.classes_))

# # stratified split
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# # ---------- 2) Oversampling MANUAL pada TRAIN ----------
# def oversample_texts(X, y, random_state=42):
#     rng = np.random.default_rng(random_state)
#     X = np.asarray(X); y = np.asarray(y)
#     counts = Counter(y); max_n = max(counts.values())
#     X_out, y_out = [], []
#     for cls, n in counts.items():
#         idx = np.where(y == cls)[0]
#         add = rng.choice(idx, size=max_n - n, replace=True) if n < max_n else np.array([], dtype=int)
#         take = np.concatenate([idx, add])
#         X_out.extend(X[take].tolist())
#         y_out.extend([cls]*len(take))
#     perm = rng.permutation(len(y_out))
#     return [X_out[i] for i in perm], np.asarray(y_out)[perm]

# X_train_os, y_train_os = oversample_texts(X_train, y_train, random_state=SEED)
# # print("Before:", Counter(y_train)); print("After:", Counter(y_train_os))

# # ---------- 3) Encode sekali (distiluse) ----------
# MODEL_NAME = "distiluse-base-multilingual-cased-v2"
# st = SentenceTransformer(MODEL_NAME)

# E_train_norm = st.encode(X_train_os, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)
# E_test_norm  = st.encode(X_test,     batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)

# E_train_raw  = st.encode(X_train_os, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)
# E_test_raw   = st.encode(X_test,     batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)

# print("Embeddings (train_os/test):", E_train_norm.shape, E_test_norm.shape)

# # ---------- 4) Helper SVD opsional ----------
# def maybe_svd_fit_transform(X, trial):
#     use_svd = trial.suggest_categorical("use_svd", [True, False])
#     if not use_svd:
#         return X, None, False
#     n_feat = X.shape[1]
#     n_comp = trial.suggest_int("svd_n_components", 64, min(512, max(64, n_feat)), step=32)
#     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
#     Xr = svd.fit_transform(X)
#     postnorm = trial.suggest_categorical("svd_post_l2norm", [True, False])
#     if postnorm:
#         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
#     return Xr, svd, postnorm

# def maybe_svd_transform(X, svd, postnorm):
#     if svd is None:
#         return X
#     Xr = svd.transform(X)
#     if postnorm:
#         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
#     return Xr

# # ---------- 5) Optuna objective (5-fold CV macro-F1) ----------
# def objective(trial: optuna.Trial) -> float:
#     # pilih normalized/non-normalized embeddings
#     use_norm = trial.suggest_categorical("use_normalized_embeddings", [True, False])
#     Xmat = E_train_norm if use_norm else E_train_raw

#     # SVD opsional
#     Xmat, svd, postnorm = maybe_svd_fit_transform(Xmat, trial)

#     # Hyperparam Logistic Regression
#     solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"])
#     class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
#     C = trial.suggest_float("C", 1e-4, 1e+4, log=True)

#     # penalty yang valid per solver
#     if solver == "lbfgs":
#         penalty, l1_ratio = "l2", None
#     elif solver == "liblinear":
#         penalty, l1_ratio = trial.suggest_categorical("penalty_liblinear", ["l1", "l2"]), None
#     else:  # saga
#         penalty = trial.suggest_categorical("penalty_saga", ["l1", "l2", "elasticnet"])
#         l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None
#         if penalty == "elasticnet" and solver != "saga":
#             raise optuna.TrialPruned()

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
#     y_tr_os = np.array(y_train_os)
#     f1s = []
#     for tr_idx, va_idx in skf.split(Xmat, y_tr_os):
#         X_tr, X_va = X


In [56]:
# # di Kaggle cell
# !pip install -q --upgrade "scikit-learn==1.5.2" "imbalanced-learn==0.12.3" "category-encoders==2.6.3"

In [57]:
# # ============================================
# # Gabung text_final + text_nostop + Oversampling + Optuna (distiluse + LogReg + SVD)
# # ============================================
# !pip -q install optuna sentence-transformers imbalanced-learn

# import numpy as np, pandas as pd, random, warnings
# warnings.filterwarnings("ignore")
# from sklearn.model_selection import train_test_split, StratifiedKFold
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sklearn.decomposition import TruncatedSVD
# from sentence_transformers import SentenceTransformer
# from imblearn.over_sampling import RandomOverSampler
# import optuna
# from optuna.pruners import MedianPruner

# SEED = 42
# random.seed(SEED); np.random.seed(SEED)

# # ---------- 1) Siapkan data ----------
# TEXT_A = "text_final"     # teks bersih (spell+normalisasi)
# TEXT_B = "text_nostop"    # versi tanpa stopwords
# LABEL  = "emotion_standardized" if "emotion_standardized" in df.columns else "emotion"
# assert TEXT_A in df.columns and TEXT_B in df.columns and LABEL in df.columns

# # Cara gabung: sederhana & aman (kedua kolom tetap dipakai)
# def combine_text(a, b):
#     a = str(a) if pd.notna(a) else ""
#     b = str(b) if pd.notna(b) else ""
#     if not a and not b: return ""
#     # gabung; gunakan separator agar tidak "nempel"
#     return f"{a} || {b}"

# df_ = df[[TEXT_A, TEXT_B, LABEL]].dropna().copy()
# df_["text_combo"] = [combine_text(a, b) for a, b in zip(df_[TEXT_A], df_[TEXT_B])]

# X_all = df_["text_combo"].astype(str).tolist()
# y_txt = df_[LABEL].astype(str).tolist()

# le = LabelEncoder()
# y_all = le.fit_transform(y_txt)
# print("Classes:", list(le.classes_))

# # split stratified (oversampling hanya pada TRAIN)
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
# )

# # ---------- 2) Oversampling (train only) ----------
# ros = RandomOverSampler(random_state=SEED)
# # imblearn butuh 2D untuk X, jadi reshape
# X_train_os, y_train_os = ros.fit_resample(np.array(X_train).reshape(-1,1), y_train)
# X_train_os = X_train_os.ravel().tolist()

# # (Opsional) lihat distribusi sesudah oversampling
# # from collections import Counter
# # print("Before:", Counter(y_train))
# # print("After :", Counter(y_train_os))

# # ---------- 3) Encode sekali (distiluse) untuk train_os & test ----------
# MODEL_NAME = "distiluse-base-multilingual-cased-v2"
# st = SentenceTransformer(MODEL_NAME)

# E_train_norm = st.encode(X_train_os, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)
# E_test_norm  = st.encode(X_test,     batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)

# E_train_raw  = st.encode(X_train_os, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)
# E_test_raw   = st.encode(X_test,     batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)

# print("Embeddings (train_os/test):", E_train_norm.shape, E_test_norm.shape)

# # ---------- 4) SVD helper (opsional via Optuna) ----------
# def maybe_svd_fit_transform(X, trial):
#     use_svd = trial.suggest_categorical("use_svd", [True, False])
#     if not use_svd:
#         return X, None, False
#     n_feat = X.shape[1]
#     n_comp = trial.suggest_int("svd_n_components", 64, min(512, max(64, n_feat)), step=32)
#     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
#     Xr = svd.fit_transform(X)
#     postnorm = trial.suggest_categorical("svd_post_l2norm", [True, False])
#     if postnorm:
#         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
#     return Xr, svd, postnorm

# def maybe_svd_transform(X, svd, postnorm):
#     if svd is None:
#         return X
#     Xr = svd.transform(X)
#     if postnorm:
#         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
#     return Xr

# # ---------- 5) Optuna objective (5-fold CV macro-F1) ----------
# def objective(trial: optuna.Trial) -> float:
#     # pilih normalized/non-normalized embeddings
#     use_norm = trial.suggest_categorical("use_normalized_embeddings", [True, False])
#     Xmat = E_train_norm if use_norm else E_train_raw

#     # SVD opsional
#     Xmat, svd, postnorm = maybe_svd_fit_transform(Xmat, trial)

#     # Hyperparam Logistic Regression
#     solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"])
#     class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
#     C = trial.suggest_float("C", 1e-4, 1e+4, log=True)

#     # penalty per solver
#     if solver == "lbfgs":
#         penalty, l1_ratio = "l2", None
#     elif solver == "liblinear":
#         penalty = trial.suggest_categorical("penalty_liblinear", ["l1", "l2"])
#         l1_ratio = None
#     else:  # saga
#         penalty = trial.suggest_categorical("penalty_saga", ["l1", "l2", "elasticnet"])
#         l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None
#         if penalty == "elasticnet" and solver != "saga":
#             raise optuna.TrialPruned()

#     # 5-fold CV di train_os (sudah balanced)
#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
#     f1s = []
#     y_tr_os = np.array(y_train_os)

#     for tr_idx, va_idx in skf.split(Xmat, y_tr_os):
#         X_tr, X_va = Xmat[tr_idx], Xmat[va_idx]
#         y_tr, y_va = y_tr_os[tr_idx], y_tr_os[va_idx]

#         clf = LogisticRegression(
#             solver=solver,
#             penalty=penalty,
#             C=C,
#             class_weight=class_weight,
#             l1_ratio=l1_ratio,
#             max_iter=3000,
#             n_jobs=-1 if solver != "liblinear" else None,
#             random_state=SEED
#         )
#         clf.fit(X_tr, y_tr)
#         y_pred = clf.predict(X_va)
#         f1s.append(f1_score(y_va, y_pred, average="macro"))

#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=50, show_progress_bar=True)

# print("\nBest trial #", study.best_trial.number)
# print("Best CV macro-F1 (oversampled train):", round(study.best_value, 6))
# print("Best params:", study.best_params)

# # ---------- 6) Train final pakai best params + evaluasi test ----------
# best = study.best_params
# use_norm = best.get("use_normalized_embeddings", True)
# X_tr_all = E_train_norm if use_norm else E_train_raw
# X_te_all = E_test_norm  if use_norm else E_test_raw

# # SVD final jika dipilih
# if best.get("use_svd", False):
#     n_comp = best["svd_n_components"]
#     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
#     X_tr_final = svd.fit_transform(X_tr_all)
#     X_te_final = svd.transform(X_te_all)
#     if best.get("svd_post_l2norm", False):
#         X_tr_final = X_tr_final / (np.linalg.norm(X_tr_final, axis=1, keepdims=True) + 1e-12)
#         X_te_final = X_te_final / (np.linalg.norm(X_te_final, axis=1, keepdims=True) + 1e-12)
# else:
#     X_tr_final, X_te_final = X_tr_all, X_te_all

# solver = best["solver"]
# class_weight = best["class_weight"]
# C = best["C"]
# if solver == "lbfgs":
#     penalty, l1_ratio = "l2", None
# elif solver == "liblinear":
#     penalty, l1_ratio = best.get("penalty_liblinear", "l2"), None
# else:
#     penalty, l1_ratio = best.get("penalty_saga", "l2"), best.get("l1_ratio", None)

# final_clf = LogisticRegression(
#     solver=solver,
#     penalty=penalty,
#     C=C,
#     class_weight=class_weight,
#     l1_ratio=l1_ratio,
#     max_iter=3000,
#     n_jobs=-1 if solver != "liblinear" else None,
#     random_state=SEED
# )
# final_clf.fit(X_tr_final, y_train_os)    # <- latih di TRAIN oversampled
# y_pred_test = final_clf.predict(X_te_final)  # <- evaluasi di TEST asli (tidak oversample)

# f1_macro = f1_score(y_test, y_pred_test, average="macro")
# f1_weighted = f1_score(y_test, y_pred_test, average="weighted")
# print("\n=== Test set (Optuna-best, train oversampled) ===")
# print("F1_macro   :", round(f1_macro, 6))
# print("F1_weighted:", round(f1_weighted, 6))
# print(classification_report(y_test, y_pred_test, target_names=le.classes_, digits=3))


In [58]:
# # ==========================================
# # A) Tambah FINE-TUNE RoBERTa (XLM-R) dkk
# # ==========================================
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, DataCollatorWithPadding
# import torch
# import numpy as np
# from sklearn.model_selection import train_test_split
# from sklearn.utils.class_weight import compute_class_weight

# # Daftar backbone yang ingin diuji (kamu bisa tambah lagi)
# FT_MODELS = [
#     "xlm-roberta-base",               # RoBERTa multilingual
#     # "indobenchmark/indobert-base-p1", # kalau mau ulang untuk pembanding
#     # "xlm-roberta-large",             # lebih akurat tapi berat (butuh VRAM besar)
# ]

# num_labels = len(le.classes_)
# SEED = 42

# # Split train->train/val untuk fine-tuning
# X_tr, X_val, y_tr, y_val = train_test_split(
#     X_train, y_train, test_size=0.1, random_state=SEED, stratify=y_train
# )

# class DS(torch.utils.data.Dataset):
#     def __init__(self, texts, labels, tokenizer, max_len=256):
#         self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len
#     def __len__(self): return len(self.texts)
#     def __getitem__(self, i):
#         enc = self.tokenizer(
#             self.texts[i],
#             truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt'
#         )
#         return {
#             "input_ids": enc["input_ids"].squeeze(0),
#             "attention_mask": enc["attention_mask"].squeeze(0),
#             "labels": torch.tensor(int(self.labels[i]), dtype=torch.long)
#         }

# for FT_MODEL in FT_MODELS:
#     print(f"\n>>> Fine-tuning backbone: {FT_MODEL}")
#     tokenizer = AutoTokenizer.from_pretrained(FT_MODEL)
#     ds_tr  = DS(X_tr,  y_tr,  tokenizer)
#     ds_val = DS(X_val, y_val, tokenizer)
#     ds_te  = DS(X_test, y_test, tokenizer)

#     # class weights (robust, handle kelas tak muncul)
#     present = np.unique(y_tr)
#     cw_present = compute_class_weight('balanced', classes=present, y=y_tr)
#     cw_map = {int(c): float(w) for c, w in zip(present, cw_present)}
#     cw_vec = [cw_map.get(i, 1.0) for i in range(num_labels)]
#     cw_t = torch.tensor(cw_vec, dtype=torch.float32)

#     model = AutoModelForSequenceClassification.from_pretrained(
#         FT_MODEL,
#         num_labels=num_labels,
#         id2label={i: l for i, l in enumerate(le.classes_)},
#         label2id={l: i for i, l in enumerate(le.classes_)}
#     )

#     args = TrainingArguments(
#         output_dir=f"./results_ft_{FT_MODEL.split('/')[-1]}",
#         num_train_epochs=3,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=16,
#         learning_rate=2e-5,
#         warmup_ratio=0.1,
#         weight_decay=0.01,
#         eval_strategy="steps",
#         eval_steps=100,
#         logging_steps=50,
#         save_steps=200,
#         load_best_model_at_end=True,
#         metric_for_best_model="eval_loss",
#         greater_is_better=False,
#         report_to="none",
#         seed=SEED
#     )

#     data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#     trainer = WeightedTrainer(
#         class_weights=cw_t,
#         model=model,
#         args=args,
#         train_dataset=ds_tr,
#         eval_dataset=ds_val,
#         data_collator=data_collator,
#         tokenizer=tokenizer
#     )

#     trainer.train()
#     pred_logits = trainer.predict(ds_te).predictions
#     y_pred = pred_logits.argmax(axis=1)
#     results.append(evaluate_preds(y_test, y_pred, f"Fine-tuned {FT_MODEL}"))


In [59]:
# # =========================================================
# # B) Tambah model Sentence-Transformers lain
# # =========================================================
# from sentence_transformers import SentenceTransformer

# # Tambah variasi encoder (pilih sesuai VRAM)
# ST_EXTRA_MODELS = [
#     "all-mpnet-base-v2",  # lebih besar & akurat
#     "all-distilroberta-v1",
#     "all-MiniLM-L12-v2"# sangat kuat multi-bahasa
#     # "intfloat/multilingual-e5-base",       # bisa dicoba; biasanya untuk retrieval, tetap oke untuk fitur
# ]

# def embed_with(model_name, texts_train, texts_test, normalize=True, bs=128):
#     st = SentenceTransformer(model_name)
#     E_tr = st.encode(texts_train, batch_size=bs, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=normalize)
#     E_te = st.encode(texts_test,  batch_size=bs, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=normalize)
#     return E_tr, E_te

# def train_3_classifiers(E_tr, y_tr, E_te):
#     from sklearn.linear_model import LogisticRegression
#     from sklearn.svm import LinearSVC
#     from xgboost import XGBClassifier
#     from sklearn.utils.class_weight import compute_class_weight
#     import numpy as np

#     cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
#     cw_dict = {int(c): float(w) for c, w in zip(np.unique(y_tr), cw)}

#     # 1) Logistic Regression
#     logreg = LogisticRegression(max_iter=1000, class_weight=cw_dict, n_jobs=-1)
#     logreg.fit(E_tr, y_tr)
#     pred_lr = logreg.predict(E_te)

#     # 2) Linear SVC
#     svc = LinearSVC()
#     svc.fit(E_tr, y_tr)
#     pred_svc = svc.predict(E_te)

#     # 3) XGBoost
#     n_classes = len(np.unique(y_train))
#     xgb = XGBClassifier(
#         objective='multi:softprob',
#         num_class=n_classes,
#         n_estimators=500,
#         learning_rate=0.05,
#         max_depth=6,
#         subsample=0.9,
#         colsample_bytree=0.9,
#         reg_lambda=1.0,
#         tree_method="hist",
#         random_state=42,
#         n_jobs=-1
#     )
#     # sample_weight untuk imbalance
#     cw_vals = compute_class_weight(class_weight='balanced', classes=np.unique(y_tr), y=y_tr)
#     cw_map = {int(c): float(w) for c, w in zip(np.unique(y_tr), cw_vals)}
#     sw = np.array([cw_map.get(int(y), 1.0) for y in y_tr], dtype=float)

#     xgb.fit(E_tr, y_tr, sample_weight=sw, verbose=False)
#     pred_xgb = xgb.predict(E_te)

#     return {
#         'LogReg': pred_lr,
#         'LinearSVC': pred_svc,
#         'XGBoost': pred_xgb
#     }

# for st_name in ST_EXTRA_MODELS:
#     print(f"\n>>> Embedding with: {st_name}")
#     E_tr, E_te = embed_with(st_name, X_train, X_test, normalize=True)
#     preds = train_3_classifiers(E_tr, y_train, E_te)
#     for clf_name, y_pred in preds.items():
#         results.append(evaluate_preds(y_test, y_pred, f"{st_name} + {clf_name}"))

# # Ringkas semua hasil lagi
# res_df = pd.DataFrame(results).sort_values("F1_macro", ascending=False).reset_index(drop=True)
# display(res_df)


In [60]:
# # !pip -q install optuna sentence-transformers

# import numpy as np
# import optuna
# from optuna.pruners import MedianPruner
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sentence_transformers import SentenceTransformer

# SEED = 42
# MODEL_NAME = "distiluse-base-multilingual-cased-v2"

# # 1) Encode sekali (normalized & non-normalized) agar cepat
# st = SentenceTransformer(MODEL_NAME)

# E_train_norm = st.encode(X_train, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)
# E_test_norm  = st.encode(X_test,  batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)

# E_train_raw  = st.encode(X_train, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)
# E_test_raw   = st.encode(X_test,  batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)

# print("Embeddings ready:", E_train_norm.shape, E_train_raw.shape)

# # 2) Objective Optuna: 5-fold CV macro-F1
# def objective(trial: optuna.Trial) -> float:
#     use_norm = trial.suggest_categorical("use_normalized_embeddings", [True, False])
#     Xtr = E_train_norm if use_norm else E_train_raw

#     solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"])
#     class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
#     C = trial.suggest_float("C", 1e-3, 1e+3, log=True)

#     # pilih penalty yang valid untuk solver
#     if solver == "lbfgs":
#         penalty = "l2"
#         l1_ratio = None
#     elif solver == "liblinear":
#         penalty = trial.suggest_categorical("penalty_liblinear", ["l1", "l2"])
#         l1_ratio = None
#     else:  # saga
#         penalty = trial.suggest_categorical("penalty_saga", ["l1", "l2", "elasticnet"])
#         l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None

#     # beberapa kombinasi tidak didukung—skip cepat
#     if solver != "saga" and penalty == "elasticnet":
#         raise optuna.TrialPruned()

#     # 5-fold CV
#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
#     f1s = []

#     for tr_idx, va_idx in skf.split(Xtr, y_train):
#         X_tr, X_va = Xtr[tr_idx], Xtr[va_idx]
#         y_tr, y_va = np.array(y_train)[tr_idx], np.array(y_train)[va_idx]

#         clf = LogisticRegression(
#             solver=solver,
#             penalty=penalty,
#             C=C,
#             class_weight=class_weight,
#             l1_ratio=l1_ratio,
#             max_iter=3000,
#             n_jobs=-1 if solver != "liblinear" else None,  # liblinear tak dukung n_jobs
#             random_state=SEED
#         )

#         clf.fit(X_tr, y_tr)
#         y_pred = clf.predict(X_va)
#         f1 = f1_score(y_va, y_pred, average="macro")
#         f1s.append(f1)

#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=0))
# study.optimize(objective, n_trials=50, show_progress_bar=True)

# print("\nBest trial:")
# print(study.best_trial)
# print("Best params:", study.best_params)

# # 3) Train final model dengan setting terbaik & evaluasi di test
# best = study.best_params
# use_norm = best.get("use_normalized_embeddings", True)
# X_tr_final = E_train_norm if use_norm else E_train_raw
# X_te_final = E_test_norm if use_norm else E_test_raw

# solver = best["solver"]
# class_weight = best["class_weight"]
# C = best["C"]

# # rekonstruksi penalty / l1_ratio sesuai solver
# if solver == "lbfgs":
#     penalty = "l2"
#     l1_ratio = None
# elif solver == "liblinear":
#     penalty = best.get("penalty_liblinear", "l2")
#     l1_ratio = None
# else:
#     penalty = best.get("penalty_saga", "l2")
#     l1_ratio = best.get("l1_ratio", None if penalty != "elasticnet" else best["l1_ratio"])

# final_clf = LogisticRegression(
#     solver=solver,
#     penalty=penalty,
#     C=C,
#     class_weight=class_weight,
#     l1_ratio=l1_ratio,
#     max_iter=3000,
#     n_jobs=-1 if solver != "liblinear" else None,
#     random_state=SEED
# )
# final_clf.fit(X_tr_final, y_train)
# y_pred_test = final_clf.predict(X_te_final)

# f1_macro = f1_score(y_test, y_pred_test, average="macro")
# f1_weighted = f1_score(y_test, y_pred_test, average="weighted")
# print("\n=== Test set results (best Optuna) ===")
# print("F1_macro  :", round(f1_macro, 6))
# print("F1_weighted:", round(f1_weighted, 6))
# print(classification_report(y_test, y_pred_test, target_names=le.classes_, digits=3))


In [61]:
# # !pip -q install optuna sentence-transformers

# import numpy as np
# import optuna
# from optuna.pruners import MedianPruner
# from sklearn.model_selection import StratifiedKFold
# from sklearn.metrics import f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sklearn.decomposition import TruncatedSVD
# from sentence_transformers import SentenceTransformer

# SEED = 42
# MODEL_NAME = "distiluse-base-multilingual-cased-v2"

# # 1) Encode sekali (normalized & non-normalized) → hemat waktu
# st = SentenceTransformer(MODEL_NAME)
# E_train_norm = st.encode(X_train, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)
# E_test_norm  = st.encode(X_test,  batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=True)
# E_train_raw  = st.encode(X_train, batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)
# E_test_raw   = st.encode(X_test,  batch_size=128, show_progress_bar=True,
#                          convert_to_numpy=True, normalize_embeddings=False)
# print("Embeddings ready:", E_train_norm.shape, E_train_raw.shape)

# # helper SVD
# def maybe_svd(X, trial, fit=True, fitted=None):
#     """
#     Jika 'use_svd' True → fit/transform TruncatedSVD.
#     Kalau False → return X apa adanya.
#     """
#     use_svd = trial.suggest_categorical("use_svd", [True, False])
#     if not use_svd:
#         return X, None

#     # komponen: pilih proporsional terhadap dimensi (umumnya 256–512 bagus untuk ST dim 512/768)
#     n_features = X.shape[1]
#     max_comp = max(8, min(512, n_features))
#     n_comp = trial.suggest_int("svd_n_components", 64, max_comp, step=32)
#     svd = fitted or TruncatedSVD(n_components=n_comp, random_state=SEED)
#     Xr = svd.fit_transform(X) if fit else svd.transform(X)

#     # opsi re-normalisasi setelah SVD (kadang bantu LR)
#     postnorm = trial.suggest_categorical("svd_post_l2norm", [True, False])
#     if postnorm:
#         Xr = Xr / (np.linalg.norm(Xr, axis=1, keepdims=True) + 1e-12)
#     return Xr, svd if fit else (Xr, fitted)

# # 2) Objective Optuna: 5-fold CV macro-F1 + (opsional) SVD
# def objective(trial: optuna.Trial) -> float:
#     use_norm = trial.suggest_categorical("use_normalized_embeddings", [True, False])
#     X_all = E_train_norm if use_norm else E_train_raw

#     solver = trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"])
#     class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])
#     C = trial.suggest_float("C", 1e-4, 1e+4, log=True)

#     # penalty valid per solver
#     if solver == "lbfgs":
#         penalty, l1_ratio = "l2", None
#     elif solver == "liblinear":
#         penalty = trial.suggest_categorical("penalty_liblinear", ["l1", "l2"])
#         l1_ratio = None
#     else:  # saga
#         penalty = trial.suggest_categorical("penalty_saga", ["l1", "l2", "elasticnet"])
#         l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0) if penalty == "elasticnet" else None
#         if penalty == "l1" and class_weight is None:
#             # (opsional): kadang l1 tanpa class_weight kurang stabil → tetap izinkan
#             pass

#     # kombinasi tidak valid → prune cepat
#     if solver != "saga" and penalty == "elasticnet":
#         raise optuna.TrialPruned()

#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
#     f1s = []

#     for tr_idx, va_idx in skf.split(X_all, y_train):
#         X_tr, X_va = X_all[tr_idx], X_all[va_idx]
#         y_tr, y_va = np.array(y_train)[tr_idx], np.array(y_train)[va_idx]

#         # (opsional) SVD fit di fold-train, apply ke val
#         X_tr_r, svd = maybe_svd(X_tr, trial, fit=True)
#         X_va_r, _   = maybe_svd(X_va, trial, fit=False, fitted=svd)

#         clf = LogisticRegression(
#             solver=solver,
#             penalty=penalty,
#             C=C,
#             class_weight=class_weight,
#             l1_ratio=l1_ratio,
#             max_iter=3000,
#             n_jobs=-1 if solver != "liblinear" else None,
#             random_state=SEED
#         )
#         clf.fit(X_tr_r, y_tr)
#         y_pred = clf.predict(X_va_r)
#         f1s.append(f1_score(y_va, y_pred, average="macro"))

#     return float(np.mean(f1s))

# study = optuna.create_study(direction="maximize", pruner=MedianPruner(n_startup_trials=10))
# study.optimize(objective, n_trials=50, show_progress_bar=True)

# print("\nBest trial:", study.best_trial.number)
# print("Best value (F1_macro CV):", study.best_value)
# print("Best params:", study.best_params)

# # 3) Train final dengan best params + evaluasi di test
# best = study.best_params
# use_norm = best.get("use_normalized_embeddings", True)
# X_tr_all = E_train_norm if use_norm else E_train_raw
# X_te_all = E_test_norm  if use_norm else E_test_raw

# # apply SVD final jika dipilih
# if best.get("use_svd", False):
#     n_comp = best["svd_n_components"]
#     svd = TruncatedSVD(n_components=n_comp, random_state=SEED)
#     X_tr_final = svd.fit_transform(X_tr_all)
#     X_te_final = svd.transform(X_te_all)
#     if best.get("svd_post_l2norm", False):
#         X_tr_final = X_tr_final / (np.linalg.norm(X_tr_final, axis=1, keepdims=True) + 1e-12)
#         X_te_final = X_te_final / (np.linalg.norm(X_te_final, axis=1, keepdims=True) + 1e-12)
# else:
#     X_tr_final, X_te_final = X_tr_all, X_te_all

# solver = best["solver"]
# class_weight = best["class_weight"]
# C = best["C"]
# if solver == "lbfgs":
#     penalty, l1_ratio = "l2", None
# elif solver == "liblinear":
#     penalty, l1_ratio = best.get("penalty_liblinear", "l2"), None
# else:
#     penalty, l1_ratio = best.get("penalty_saga", "l2"), best.get("l1_ratio", None)

# final_clf = LogisticRegression(
#     solver=solver,
#     penalty=penalty,
#     C=C,
#     # class_weight=class_weight,
#     l1_ratio=l1_ratio,
#     max_iter=3000,
    
#     n_jobs=-1 if solver != "liblinear" else None,
#     random_state=SEED
# )
# final_clf.fit(X_tr_final, y_train)
# y_pred_test = final_clf.predict(X_te_final)

# f1_macro = f1_score(y_test, y_pred_test, average="macro")
# f1_weighted = f1_score(y_test, y_pred_test, average="weighted")
# print("\n=== Test set results (Optuna + SVD option) ===")
# print("F1_macro   :", round(f1_macro, 6))
# print("F1_weighted:", round(f1_weighted, 6))
# print(classification_report(y_test, y_pred_test, target_names=le.classes_, digits=3))


In [62]:
# import torch
# from transformers import Trainer

# class FocalLoss(torch.nn.Module):
#     def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
#         super().__init__()
#         self.alpha = alpha  # tensor class weights or None
#         self.gamma = gamma
#         self.reduction = reduction
#         self.ce = torch.nn.CrossEntropyLoss(weight=alpha, reduction='none')

#     def forward(self, logits, target):
#         ce = self.ce(logits, target)               # [B]
#         pt = torch.softmax(logits, dim=-1)[range(len(target)), target].clamp_min(1e-8)
#         loss = ((1 - pt) ** self.gamma) * ce
#         return loss.mean() if self.reduction == 'mean' else loss.sum()

# class FocalTrainer(Trainer):
#     def __init__(self, class_weights=None, gamma=2.0, **kwargs):
#         super().__init__(**kwargs)
#         alpha = None
#         if class_weights is not None:
#             alpha = class_weights if isinstance(class_weights, torch.Tensor) else torch.tensor(class_weights, dtype=torch.float32)
#         self.focal = FocalLoss(alpha=alpha, gamma=gamma)

#     def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
#         labels = inputs.pop("labels")
#         outputs = model(**inputs)
#         loss = self.focal(outputs.logits, labels)
#         return (loss, outputs) if return_outputs else loss


In [63]:
# # cw_t = tensor panjang num_labels (sudah kamu buat sebelumnya)
# trainer = FocalTrainer(
#     class_weights=cw_t, gamma=2.0,
#     model=model, args=args,
#     train_dataset=ds_tr, eval_dataset=ds_val,
#     data_collator=data_collator, tokenizer=tokenizer
# )
# trainer.train()


In [64]:
# from torch.utils.data import DataLoader, WeightedRandomSampler
# import numpy as np

# # hitung weight per sample (kebalikan frekuensi kelas)
# counts = np.bincount(y_tr)
# class_w = 1.0 / np.maximum(counts, 1)
# sample_w = class_w[y_tr]
# sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

# train_loader = DataLoader(ds_tr, batch_size=16, sampler=sampler)
# eval_loader  = DataLoader(ds_val, batch_size=16, shuffle=False)

# # Pass custom dataloaders ke Trainer:
# trainer = FocalTrainer(
#     class_weights=cw_t, gamma=2.0,
#     model=model, args=args, tokenizer=tokenizer,
#     train_dataset=None, eval_dataset=None,  # set None
#     data_collator=data_collator
# )
# trainer.get_train_dataloader = lambda: train_loader
# trainer.get_eval_dataloader  = lambda eval_dataset=None: eval_loader

# trainer.train()
